### Cell 1 (Markdown)

# Macro & Markets Lab
*Five free data sources, one research pipeline: Treasury curves, payrolls and oil inventories on one side; SPY trend signals, crypto momentum and energy stocks on the other.*

| Module | Question | Data |
|---|---|---|
| **A. Backtester** | Does a 50/200-day SMA crossover beat buy-and-hold SPY after costs? | Alpha Vantage (fallback: yfinance) |
| **B. Macro dashboard** | Where are the yield curve, inflation and unemployment relative to past recessions? | FRED |
| **C. Crypto screener** | Which large coins combine strong momentum with heavy turnover? | CoinGecko |
| **D. Energy study** | How closely do XLE/XOM track oil and gas, and do inventory surprises predict next-week returns? | EIA + prices |
| **E. Labor indicator** | One robust z-score for labor-market momentum | BLS |
| **F. Regime study** | How did the trend strategy, SPY, QQQ and TLT do in expansion, slowdown and contraction? | FRED + BLS + EIA + prices |

**To run:** add whichever keys you have in Colab's 🔑 **Secrets** panel (names are listed in Cell 3), turn on *Notebook access*, then *Runtime ▸ Run all*. Every source has a keyless fallback, so the notebook also runs end-to-end with no keys at all. Keys only buy you the primary sources and higher limits.

**Look-ahead controls used throughout**
- Trading signals use day *t*'s close and execute at day *t+1*'s open.
- Monthly macro data are shifted by their publication delay (CPI and jobs data: one month) before they drive any label.
- Weekly EIA inventories are matched only to returns that start *after* the Wednesday release.
- Z-scores use expanding windows (past data only). NBER recession dates are used for shading and evaluation, never as an input.

In [ ]:
# ### Cell 2 (Code)
# Colab ships an old yfinance (0.2.x); Yahoo changes often, so upgrade before importing it.
# If yfinance was already imported in this session, restart the runtime after this cell.
%pip install -q --upgrade "yfinance>=1.5.2"
%pip install -q plotly statsmodels xlrd requests

import hashlib
import json
import os
import random
import re
import time
import warnings
from io import BytesIO, StringIO
from pathlib import Path
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import requests
import statsmodels.api as sm
import yfinance as yf
from IPython.display import Markdown, display
from pandas.tseries.holiday import USFederalHolidayCalendar
from plotly.subplots import make_subplots
from scipy import stats

try:
    from google.colab import userdata  # Colab Secrets API
    IN_COLAB = True
except ImportError:
    userdata, IN_COLAB = None, False

pio.templates.default = "plotly_white"
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"pandas {pd.__version__} | numpy {np.__version__} | yfinance {yf.__version__} | Colab: {IN_COLAB}")

In [ ]:
# ### Cell 3 (Code)
# ── Universe & sample ────────────────────────────────────────────────────────
EQUITY_TICKERS = ["SPY", "QQQ", "TLT", "XLE", "XOM"]
PRICE_START = "2000-01-01"
MACRO_START = "1976-06-01"   # first day of the 10Y–2Y series: z-score warm-up and every inversion since 1978
END_DATE = pd.Timestamp.now(tz="America/New_York").normalize().tz_localize(None)   # exchange date, not the machine's

# ── Module A: SMA crossover backtest ─────────────────────────────────────────
FAST_MA, SLOW_MA = 50, 200
COST_BPS = 5.0                       # one-way cost per 100% of NAV traded (commission + spread + slippage)
COST_GRID_BPS = [0, 2, 5, 10, 25, 50]
CASH_EARNS_RISK_FREE = True          # when out of the market the strategy earns the fed funds rate
TRADING_DAYS = 252

# ── Module D: energy study ───────────────────────────────────────────────────
ENERGY_CORR_START = "2008-01-01"     # Henry Hub daily spot has large gaps before 2008
CORR_WINDOWS = [63, 252]             # ~3 months and ~1 year of trading days
INVENTORY_HOLD_DAYS = 5              # "next week" = 5 trading days
# Weekly Petroleum Status Report: Wednesday 10:30 ET for the week ending the prior Friday; a federal holiday
# Mon–Wed pushes it to Thursday. Irregular delays that the holiday rule can't infer:
WPSR_RELEASE_OVERRIDES = {"2025-12-19": "2025-12-29"}   # week ending → actual release (Christmas 2025, 5 pm ET)

# ── Modules E/F: when each monthly print became public ───────────────────────
RELEASE_LAG_MONTHS = {
    "cpi": 1,         # CPI for month m comes out mid-month m+1
    "labor": 1,       # payrolls, unemployment and wages: Employment Situation, first Friday of m+1
    "fed_funds": 1,   # FEDFUNDS is a monthly average, published after the month closes
    "curve": 0,       # Treasury yields are observed daily, so the month's average is known at month-end
    "oil": 0,         # spot oil trades daily
}
# Reference month → actual release date, where the normal lag was broken (2025 federal shutdown)
RELEASE_DATE_OVERRIDES = {"labor": {"2025-09": "2025-11-20", "2025-10": "2025-12-16", "2025-11": "2025-12-16"}}
ZSCORE_MIN_MONTHS = 36
WAGE_SERIES = "CES0500000008"        # avg hourly earnings, production & nonsupervisory (from 1964); "CES0500000003" = all employees (from 2006)

# ── Price-source policy ─────────────────────────────────────────────────────
REQUIRE_ADJUSTED_PRICES = True       # TLT's return is mostly coupons; unadjusted closes would badly understate buy-and-hold
MIN_PRICE_ROWS = 1000                # Alpha Vantage "compact" output (100 rows) can't support a 200-day MA backtest

# ── Cache & HTTP ─────────────────────────────────────────────────────────────
PERSIST_CACHE_TO_DRIVE = False       # True keeps the cache across Colab runtime resets
CACHE_TTL_HOURS = {"prices": 24, "fred": 24, "eia": 24, "bls": 24, "coingecko": 1}
NEGATIVE_CACHE_HOURS = 24            # remember "premium endpoint" / "daily cap" refusals so reruns don't spend quota
MAX_RETRIES = 5
BACKOFF_BASE_SEC = 2.0
MAX_RETRY_AFTER_SEC = 90             # a longer Retry-After means "come back later": use the fallback instead of waiting
REQUEST_TIMEOUT_SEC = 30

if PERSIST_CACHE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/macro_lab_cache")
else:
    CACHE_DIR = Path("/content/macro_lab_cache") if IN_COLAB else Path("macro_lab_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── API keys: Colab ▸ 🔑 Secrets ▸ "Add new secret" ▸ enable Notebook access ─
# Never paste keys into code. All five are free:
#   ALPHAVANTAGE_API_KEY  https://www.alphavantage.co/support/#api-key   (without it: yfinance)
#   FRED_API_KEY          https://fredaccount.stlouisfed.org/apikeys     (without it: keyless fredgraph.csv)
#   EIA_API_KEY           https://www.eia.gov/opendata/register.php      (without it: FRED mirrors + EIA history file)
#   BLS_API_KEY           https://data.bls.gov/registrationEngine/       (without it: keyless BLS v1, then FRED mirrors)
#   COINGECKO_API_KEY     https://www.coingecko.com/en/api/pricing       ("Demo" plan; without it: keyless public API)
API_KEY_NAMES = ["ALPHAVANTAGE_API_KEY", "FRED_API_KEY", "EIA_API_KEY", "BLS_API_KEY", "COINGECKO_API_KEY"]


def get_secret(name):
    """Colab Secrets first, then environment variables (for local runs)."""
    if IN_COLAB:
        try:
            value = userdata.get(name)
            if value:
                return value.strip()
        except Exception as exc:  # SecretNotFoundError / NotebookAccessError / TimeoutException
            kind = type(exc).__name__
            if "Access" in kind:
                print(f"  {name}: exists but Notebook access is off — toggle it in the Secrets panel")
            elif "Timeout" in kind:
                print(f"  {name}: Colab didn't answer within 10 s (access prompt not confirmed?) — rerun this cell")
    value = os.environ.get(name, "").strip()
    return value or None


API_KEYS = {name: get_secret(name) for name in API_KEY_NAMES}
print(pd.DataFrame({"secret": API_KEY_NAMES,
                    "status": ["loaded" if API_KEYS[n] else "missing → keyless fallback" for n in API_KEY_NAMES]}).to_string(index=False))
print(f"\nCache directory: {CACHE_DIR.resolve()}   |   sample: prices from {PRICE_START}, macro from {MACRO_START}")

### Cell 4 (Markdown)

## Data layer: caching, retries and provenance
Free tiers are tight (Alpha Vantage allows 25 requests a day, keyless BLS 25 queries a day), so every request goes through one `fetch()` function that:
1. **Caches to disk** under `CACHE_DIR`, keyed by URL and parameters with API keys stripped out, with a TTL per source. A rerun inside the TTL makes no network calls.
2. **Never caches failures.** Payloads are validated before they are saved; Alpha Vantage, for example, answers HTTP 200 with a "premium endpoint" note instead of data. Permanent refusals are remembered for `NEGATIVE_CACHE_HOURS` in a small `.err` marker, so a rerun doesn't spend quota asking again.
3. **Retries with exponential backoff and jitter** on HTTP 429/5xx, timeouts and rate-limit notes, and honours `Retry-After`.
4. **Falls back to a stale cached copy** if the network fails and an older copy exists.
5. **Logs provenance** (which source was actually used, row count, date span), so you can see which fallbacks fired.

In [ ]:
# ### Cell 5 (Code)
SECRET_FIELDS = {"apikey", "api_key", "registrationkey", "x_cg_demo_api_key", "x-cg-demo-api-key"}
RETRYABLE_STATUS = {429, 500, 502, 503, 504}
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "macro-markets-lab/1.0 (research notebook; python-requests)"
_LAST_CALL = {}
DATA_LOG = []


class PermanentAPIError(Exception):
    """A well-formed answer we can't use (premium endpoint, daily cap, bad key or series). Don't retry."""


class TransientAPIError(Exception):
    """Rate limit, timeout or server error. Retry with backoff."""

    def __init__(self, message, retry_after=None):
        super().__init__(message)
        self.retry_after = retry_after


def redact(text):
    text = str(text)
    for key in filter(None, API_KEYS.values()):
        text = text.replace(key, "***")
    return text


def _public(mapping):
    return {k: v for k, v in sorted((mapping or {}).items()) if k.lower() not in SECRET_FIELDS}


def _secret_fingerprint(*mappings):
    secrets = [str(v) for m in mappings for k, v in sorted((m or {}).items()) if k.lower() in SECRET_FIELDS]
    return hashlib.sha256("|".join(secrets).encode()).hexdigest()[:8]


def _cache_paths(namespace, url, params, body, headers, fmt):
    blob = json.dumps([url, _public(params), _public(body)], sort_keys=True, default=str)
    stem = hashlib.sha256(blob.encode()).hexdigest()[:24]
    folder = CACHE_DIR / namespace
    suffix = {"json": "json", "text": "txt", "bytes": "bin"}[fmt]
    # Data doesn't depend on which key fetched it, but an error ("invalid key") does.
    return folder / f"{stem}.{suffix}", folder / f"{stem}_{_secret_fingerprint(params, body, headers)}.err"


def _read_cache(path, fmt):
    if fmt == "json":
        return json.loads(path.read_text(encoding="utf-8"))
    return path.read_text(encoding="utf-8") if fmt == "text" else path.read_bytes()


def _write_cache(path, payload, fmt):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    if fmt == "json":
        tmp.write_text(json.dumps(payload), encoding="utf-8")
    elif fmt == "text":
        tmp.write_text(payload, encoding="utf-8")
    else:
        tmp.write_bytes(payload)
    os.replace(tmp, path)  # atomic, so a crash mid-write never leaves a half-written cache file


def _is_fresh(path, ttl_hours):
    return path.exists() and (ttl_hours is None or (time.time() - path.stat().st_mtime) / 3600 < ttl_hours)


def _retry_after_seconds(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _throttle(namespace, min_interval_sec):
    wait = _LAST_CALL.get(namespace, 0.0) + min_interval_sec - time.time()
    if wait > 0:
        time.sleep(wait)
    _LAST_CALL[namespace] = time.time()


def with_retries(call, label):
    """Run call() with exponential backoff + jitter on transient failures."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return call()
        except (TransientAPIError, requests.ConnectionError, requests.Timeout,
                requests.exceptions.ChunkedEncodingError) as exc:
            hinted = _retry_after_seconds(getattr(exc, "retry_after", None))
            if hinted and hinted > MAX_RETRY_AFTER_SEC:  # e.g. a quota that resets at midnight: don't sit and wait
                raise TransientAPIError(f"{label}: rate-limited, server says retry in {hinted:.0f}s", retry_after=hinted) from None
            if attempt == MAX_RETRIES:
                raise TransientAPIError(f"{label}: gave up after {MAX_RETRIES} attempts ({redact(exc)[:160]})") from None
            wait = hinted if hinted else min(BACKOFF_BASE_SEC * 2 ** (attempt - 1), 60.0) * random.uniform(1.0, 1.25)
            print(f"   ↻ {label}: {redact(exc)[:110]} — retry {attempt}/{MAX_RETRIES - 1} in {wait:.0f}s")
            time.sleep(wait)


def _stale_or_raise(path, fmt, namespace, exc):
    if path.exists():
        stamp = pd.Timestamp(path.stat().st_mtime, unit="s").strftime("%Y-%m-%d %H:%M")
        print(f"   ⚠️ {namespace}: {redact(exc)[:110]} — using stale cache from {stamp} UTC")
        return _read_cache(path, fmt)
    raise exc


def fetch(url, *, namespace, params=None, json_body=None, headers=None, fmt="json",
          ttl_hours=24, validate=None, min_interval_sec=0.0, literal_brackets=False):
    """GET (or POST when json_body is given) with disk cache, retries and payload validation."""
    path, err_path = _cache_paths(namespace, url, params, json_body, headers, fmt)
    if _is_fresh(path, ttl_hours):
        return _read_cache(path, fmt)
    # EIA documents its query syntax with literal brackets (data[0]=value); requests would send %5B%5D.
    query = urlencode(params, safe="[]") if (params and literal_brackets) else params

    def _call():
        _throttle(namespace, min_interval_sec)
        method = "POST" if json_body is not None else "GET"
        resp = SESSION.request(method, url, params=query, json=json_body, headers=headers, timeout=REQUEST_TIMEOUT_SEC)
        if resp.status_code in RETRYABLE_STATUS:
            raise TransientAPIError(f"HTTP {resp.status_code}", resp.headers.get("Retry-After"))
        if resp.status_code >= 400:
            raise PermanentAPIError(f"HTTP {resp.status_code}: {redact(resp.text[:240])}")
        if fmt == "json":
            try:
                payload = resp.json()
            except ValueError:
                raise TransientAPIError(f"non-JSON reply: {resp.text[:80]!r}") from None
        else:
            payload = resp.text if fmt == "text" else resp.content
        if validate is not None:
            validate(payload)
        return payload

    remembered = _is_fresh(err_path, NEGATIVE_CACHE_HOURS)
    try:
        if remembered:
            raise PermanentAPIError(err_path.read_text(encoding="utf-8") + " [remembered from an earlier run]")
        payload = with_retries(_call, namespace)
    except PermanentAPIError as exc:
        if not remembered:
            err_path.parent.mkdir(parents=True, exist_ok=True)
            err_path.write_text(redact(exc), encoding="utf-8")
        return _stale_or_raise(path, fmt, namespace, exc)
    except TransientAPIError as exc:
        wait = _retry_after_seconds(exc.retry_after) or 0
        if wait > MAX_RETRY_AFTER_SEC:  # remember the refusal until the server's window reopens
            err_path.parent.mkdir(parents=True, exist_ok=True)
            err_path.write_text(redact(exc), encoding="utf-8")
            expires_in = min(wait, NEGATIVE_CACHE_HOURS * 3600)
            stamp = time.time() - NEGATIVE_CACHE_HOURS * 3600 + expires_in
            os.utime(err_path, (stamp, stamp))
        return _stale_or_raise(path, fmt, namespace, exc)
    _write_cache(path, payload, fmt)
    err_path.unlink(missing_ok=True)
    return payload


def cached_frame(namespace, name, loader, ttl_hours=24):
    """Disk-cache a DataFrame produced by a library call (e.g. yfinance) as CSV, with the same retry rules."""
    path = CACHE_DIR / namespace / f"{re.sub(r'[^A-Za-z0-9_.-]', '_', name)}.csv"
    if _is_fresh(path, ttl_hours):
        return pd.read_csv(path, index_col=0, parse_dates=True)
    try:
        df = with_retries(loader, f"{namespace}:{name}")
    except TransientAPIError as exc:
        if path.exists():
            print(f"   ⚠️ {namespace}:{name}: {exc} — using stale cache")
            return pd.read_csv(path, index_col=0, parse_dates=True)
        raise
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".csv.tmp")
    df.to_csv(tmp)
    os.replace(tmp, path)
    return df


def month_end(index):
    return pd.DatetimeIndex(index) + pd.offsets.MonthEnd(0)


def log_source(dataset, source, data, note=""):
    obj = data.dropna(how="all") if isinstance(data, pd.DataFrame) else data.dropna()
    dated = isinstance(obj.index, pd.DatetimeIndex) and len(obj)
    DATA_LOG.append({"dataset": dataset, "source": source, "rows": len(obj),
                     "first": obj.index.min().date() if dated else None,
                     "last": obj.index.max().date() if dated else None, "note": note})

### Cell 6 (Markdown)

## Data sources and alignment conventions
| Source | What | Frequency | Primary → fallback |
|---|---|---|---|
| Alpha Vantage | SPY, QQQ, TLT, XLE, XOM OHLCV | daily | `TIME_SERIES_DAILY_ADJUSTED` (`TIME_SERIES_DAILY` if unadjusted prices are allowed) → yfinance, auto-adjusted. On a **free** key both full-history calls are premium-only (since late 2025) and "compact" holds only 100 days, so expect the yfinance fallback; a premium key uses AV directly. |
| FRED | DGS10, DGS2, T10Y2Y, CPIAUCSL, UNRATE, FEDFUNDS, USREC | daily / monthly | API → keyless `fredgraph.csv` |
| CoinGecko | top-100 coins: price, market cap, volume, 24h and 7d % change | snapshot | demo key → keyless public API |
| EIA | WTI Cushing spot (RWTC), Henry Hub spot (RNGWHHD), commercial crude stocks ex-SPR (WCESTUS1) | daily / weekly | API v2 → FRED mirrors (prices) / EIA history file (stocks) |
| BLS | nonfarm payrolls (CES0000000001), unemployment rate (LNS14000000), avg hourly earnings of production & nonsupervisory employees (CES0500000008, set by `WAGE_SERIES`) | monthly | API v2 → keyless v1 → FRED mirrors (PAYEMS, UNRATE, AHETPI) |

**Calendar conventions.** Daily data sit on the trading calendar of the price files, and macro dailies are inner-joined onto it. Weekly EIA data keep their week-ending (Friday) date plus an explicit release date. Monthly data are stamped at **month-end**, and Module F shifts each series by its release lag before using it.

In [ ]:
# ### Cell 7 (Code)
AV_URL = "https://www.alphavantage.co/query"
AV_MIN_INTERVAL_SEC = 1.5            # free tier: 1 request/second burst limit and 25 requests/day
_av_state = {"blocked": None, "premium": set()}   # session-wide: daily cap / bad key, and functions refused as premium


def validate_alphavantage(payload):
    """AV answers HTTP 200 for everything, so success is detected by the presence of a 'Time Series' key.
    Order matters: the burst-limit note also mentions 'premium endpoints'."""
    if isinstance(payload, dict) and any(k.startswith("Time Series") for k in payload):
        return
    if not isinstance(payload, dict):
        raise PermanentAPIError("Alpha Vantage: unexpected payload")
    message = redact(" ".join(str(payload[k]) for k in ("Information", "Note", "Error Message") if k in payload)
                     or str(payload)[:200])
    text = message.lower()
    if "demo" in text:
        raise PermanentAPIError(f"demo key: {message[:120]}")
    if "is a premium endpoint" in text or "premium feature" in text:
        raise PermanentAPIError(f"premium-only on the free tier: {message[:150]}")
    if "sparingly" in text or "per second" in text or "calls per minute" in text:
        raise TransientAPIError(f"burst rate limit: {message[:120]}", retry_after=3)
    if "per day" in text:
        raise PermanentAPIError(f"daily request cap reached: {message[:150]}")
    raise PermanentAPIError(message[:200])  # invalid key / invalid call


def parse_alphavantage_daily(payload):
    series = next(v for k, v in payload.items() if k.startswith("Time Series"))
    raw = pd.DataFrame.from_dict(series, orient="index").apply(pd.to_numeric, errors="coerce")
    raw.columns = [re.sub(r"^\d+\.\s*", "", c).strip().replace(" ", "_") for c in raw.columns]
    raw.index = pd.to_datetime(raw.index)
    raw = raw.sort_index()
    adjusted = "adjusted_close" in raw
    factor = raw["adjusted_close"] / raw["close"] if adjusted else 1.0  # rescales OHLC for splits and dividends
    out = raw[["open", "high", "low", "close"]].mul(factor, axis=0)
    out["volume"] = raw["volume"]
    out.index.name = "date"
    return out.loc[PRICE_START:END_DATE - pd.Timedelta(days=1)], adjusted  # today's bar may still be provisional


def load_prices_alphavantage(ticker):
    key = API_KEYS["ALPHAVANTAGE_API_KEY"]
    if not key:
        return None, "no ALPHAVANTAGE_API_KEY secret"
    if _av_state["blocked"]:
        return None, _av_state["blocked"]
    functions = ["TIME_SERIES_DAILY_ADJUSTED"] + ([] if REQUIRE_ADJUSTED_PRICES else ["TIME_SERIES_DAILY"])
    reasons = []
    for function in functions:
        if function in _av_state["premium"]:
            reasons.append(f"{function}: premium-only (refused earlier this run)")
            continue
        try:
            payload = fetch(AV_URL, namespace="alphavantage",
                            params={"function": function, "symbol": ticker, "outputsize": "full", "apikey": key},
                            ttl_hours=CACHE_TTL_HOURS["prices"], validate=validate_alphavantage,
                            min_interval_sec=AV_MIN_INTERVAL_SEC)
        except (PermanentAPIError, TransientAPIError) as exc:
            reasons.append(f"{function}: {exc}")
            text = str(exc).lower()
            if "premium-only" in text:
                _av_state["premium"].add(function)  # the same answer applies to every symbol
            if isinstance(exc, TransientAPIError) or any(s in text for s in ("daily request cap", "apikey is invalid", "demo key")):
                _av_state["blocked"] = str(exc)  # no point trying the other tickers today
                break
            continue
        df, adjusted = parse_alphavantage_daily(payload)
        if len(df) < MIN_PRICE_ROWS:
            reasons.append(f"{function}: only {len(df)} rows (compact output)")
            continue
        return df, f"Alpha Vantage {function}" + ("" if adjusted else " (unadjusted)")
    return None, " | ".join(reasons)


try:  # yfinance keeps a small SQLite tz cache; keep it on local disk even when CACHE_DIR is on Google Drive
    yf.set_tz_cache_location(str(Path("/content/py-yfinance") if IN_COLAB else CACHE_DIR / "py-yfinance"))
except Exception:
    pass


def load_prices_yfinance(ticker):
    def download():
        try:
            # `end` is exclusive: stopping at today avoids an in-progress intraday bar posing as a daily close.
            hist = yf.Ticker(ticker).history(start=PRICE_START, end=END_DATE, auto_adjust=True, actions=False)
        except Exception as exc:
            if "rate" in type(exc).__name__.lower() or "too many" in str(exc).lower():
                raise TransientAPIError(f"Yahoo rate limit: {exc}", retry_after=30) from None
            raise
        if hist.empty:
            raise TransientAPIError(f"yfinance returned no rows for {ticker} (Yahoo throttling?)")
        hist.index = pd.DatetimeIndex(hist.index.tz_localize(None).normalize(), name="date")
        return hist.rename(columns=str.lower)[["open", "high", "low", "close", "volume"]]

    return cached_frame("yfinance", f"{ticker}_{PRICE_START}", download, ttl_hours=CACHE_TTL_HOURS["prices"])


def load_all_prices(tickers):
    prices, sources = {}, {}
    for ticker in tickers:
        try:
            df, info = load_prices_alphavantage(ticker)
            if df is None:
                print(f"⚠️ {ticker}: Alpha Vantage unavailable ({info[:160]}) → falling back to yfinance")
                df, info = load_prices_yfinance(ticker), "yfinance (auto-adjusted)"
        except Exception as exc:  # one throttled or mistyped ticker shouldn't take the others down with it
            print(f"❌ {ticker}: no price source worked ({redact(exc)[:160]}) — modules that need it will be skipped")
            continue
        prices[ticker], sources[ticker] = df.sort_index(), info
        log_source(f"{ticker} daily OHLCV", info, df["close"])
    if any("unadjusted" in s for s in sources.values()):
        print("⚠️ Some prices are split/dividend-unadjusted: buy-and-hold returns (TLT and XOM especially) are understated.")
    return prices

In [ ]:
# ### Cell 8 (Code)
FRED_API_URL = "https://api.stlouisfed.org/fred/series/observations"
FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv"
FRED_SERIES = {
    "DGS10": "10-year Treasury yield (%)",
    "DGS2": "2-year Treasury yield (%)",
    "T10Y2Y": "10Y minus 2Y spread (pp)",
    "CPIAUCSL": "CPI-U all items, SA (index)",
    "UNRATE": "Unemployment rate (%)",
    "FEDFUNDS": "Effective fed funds rate, monthly avg (%)",
    "USREC": "NBER recession indicator (1 = recession)",
}


def validate_fred_json(payload):
    if isinstance(payload, dict) and payload.get("observations"):  # an empty list is not usable data either
        return
    detail = payload.get("error_message", payload) if isinstance(payload, dict) else payload
    raise PermanentAPIError(f"FRED: {str(detail)[:200]}")


def validate_fred_csv(text):
    header = text.lstrip()[:40].lower()
    if not (header.startswith("observation_date") or header.startswith("date")):
        raise PermanentAPIError(f"FRED CSV: unexpected reply {text[:80]!r}")


def fred_from_api(series_id, start):
    payload = fetch(FRED_API_URL, namespace="fred",
                    params={"series_id": series_id, "api_key": API_KEYS["FRED_API_KEY"],
                            "file_type": "json", "observation_start": start},
                    ttl_hours=CACHE_TTL_HOURS["fred"], validate=validate_fred_json, min_interval_sec=0.6)
    obs = pd.DataFrame(payload["observations"])
    return pd.Series(pd.to_numeric(obs["value"], errors="coerce").to_numpy(),  # FRED encodes missing values as "."
                     index=pd.to_datetime(obs["date"]), name=series_id)


def fred_from_csv(series_id, start):
    text = fetch(FRED_CSV_URL, namespace="fred_csv", params={"id": series_id, "cosd": start}, fmt="text",
                 ttl_hours=CACHE_TTL_HOURS["fred"], validate=validate_fred_csv, min_interval_sec=0.5)
    df = pd.read_csv(StringIO(text))
    s = pd.Series(pd.to_numeric(df.iloc[:, 1], errors="coerce").to_numpy(),
                  index=pd.to_datetime(df.iloc[:, 0]), name=series_id)
    return s.loc[start:]


def load_fred_series(series_id, start=MACRO_START, label=None):
    s, source = None, None
    if API_KEYS["FRED_API_KEY"]:
        try:
            s, source = fred_from_api(series_id, start), "FRED API"
        except (PermanentAPIError, TransientAPIError) as exc:
            print(f"⚠️ FRED API failed for {series_id} ({exc}) → keyless CSV")
    if s is None:
        s, source = fred_from_csv(series_id, start), "FRED fredgraph.csv (keyless)"
    s = s[~s.index.duplicated(keep="last")].sort_index()
    log_source(label or f"FRED {series_id}: {FRED_SERIES.get(series_id, '')}", source, s)
    return s

In [ ]:
# ### Cell 9 (Code)
COINGECKO_URL = "https://api.coingecko.com/api/v3"   # keyless and Demo keys; paid keys use pro-api.coingecko.com
COINGECKO_MIN_INTERVAL_SEC = 2.5     # keyless limits are per (shared) IP, so stay gentle
# Exclusions are by CoinGecko id, not symbol: symbols collide (e.g. two different "USDC" tokens).
# Since Feb 2026 /coins/markets already omits wrapped, bridged and liquid-staked tokens by default.
STATIC_EXCLUSIONS = {
    "stablecoin": {"tether", "usd-coin", "usds", "ethena-usde", "dai", "usd1-wlfi", "global-dollar", "paypal-usd",
                   "ripple-usd", "falcon-finance", "usdd", "united-stables", "bfusd", "usdgo", "open-usd", "gho",
                   "first-digital-usd", "true-usd", "ethena-staked-usde", "susds", "usdt0", "frax", "euro-coin"},
    "tokenized fund / gold": {"figure-heloc", "hashnote-usyc", "ondo-us-dollar-yield", "tether-gold", "pax-gold",
                              "blackrock-usd-institutional-digital-liquidity-fund", "blockchain-capital",
                              "spiko-amundi-overnight-swap-fund-eur",
                              "superstate-short-duration-us-government-securities-fund-ustb"},
    "wrapped / staked": {"wrapped-bitcoin", "weth", "staked-ether", "wrapped-steth", "coinbase-wrapped-btc",
                         "wrapped-eeth", "wrapped-beacon-eth", "binance-peg-weth", "jito-staked-sol",
                         "binance-staked-sol", "wbnb", "clbtc", "staked-trx"},
}


def validate_coingecko_list(payload):
    if isinstance(payload, list):
        return
    raise PermanentAPIError(f"CoinGecko: {str(payload)[:200]}")


_cg_state = {"use_key": bool(API_KEYS["COINGECKO_API_KEY"])}


def coingecko_get(path, params, ttl_hours):
    """Demo key if available; a rejected key (wrong plan, typo, revoked) falls back to the keyless API."""
    def call(with_key):
        headers = {"x-cg-demo-api-key": API_KEYS["COINGECKO_API_KEY"]} if with_key else None
        return fetch(f"{COINGECKO_URL}/{path}", namespace="coingecko", params=params, headers=headers,
                     ttl_hours=ttl_hours, validate=validate_coingecko_list, min_interval_sec=COINGECKO_MIN_INTERVAL_SEC)

    if _cg_state["use_key"]:
        try:
            return call(True)
        except PermanentAPIError as exc:
            print(f"⚠️ CoinGecko rejected the demo key ({str(exc)[:100]}) → keyless public API")
            _cg_state["use_key"] = False
    return call(False)


def load_coingecko_markets(n=100):
    payload = coingecko_get("coins/markets", {"vs_currency": "usd", "order": "market_cap_desc", "per_page": n,
                                              "page": 1, "price_change_percentage": "24h,7d", "sparkline": "false"},
                            CACHE_TTL_HOURS["coingecko"])
    df = pd.DataFrame(payload).rename(columns={"price_change_percentage_24h_in_currency": "chg_24h",
                                               "price_change_percentage_7d_in_currency": "chg_7d"})
    df = df[["id", "symbol", "name", "market_cap_rank", "current_price", "market_cap", "total_volume",
             "chg_24h", "chg_7d", "last_updated"]].copy()
    df["symbol"] = df["symbol"].str.upper()
    for col in ["current_price", "market_cap", "total_volume", "chg_24h", "chg_7d"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    source = "CoinGecko " + ("demo key" if _cg_state["use_key"] else "keyless public API")
    log_source("CoinGecko top-100 snapshot", source, df, note=f"as of {df['last_updated'].max()}")
    return df


def load_excluded_coin_ids():
    """Coin id → reason: CoinGecko's own 'stablecoins' category (top 250 by cap) merged with STATIC_EXCLUSIONS."""
    excluded = {coin: reason for reason, ids in STATIC_EXCLUSIONS.items() for coin in ids}
    try:
        rows = coingecko_get("coins/markets", {"vs_currency": "usd", "category": "stablecoins",
                                               "order": "market_cap_desc", "per_page": 250, "page": 1}, ttl_hours=24)
        excluded.update({row["id"]: "stablecoin" for row in rows})
    except (PermanentAPIError, TransientAPIError) as exc:
        print(f"⚠️ CoinGecko stablecoin category unavailable ({str(exc)[:100]}); using the static list only")
    return excluded

In [ ]:
# ### Cell 10 (Code)
EIA_URL = "https://api.eia.gov/v2"
EIA_PAGE_ROWS = 5000                 # API v2 hard cap per request; we page with `offset`
EIA_SERIES = {
    "wti": {"route": "petroleum/pri/spt", "series": "RWTC", "frequency": "daily", "fred": "DCOILWTICO",
            "label": "EIA WTI Cushing spot ($/bbl)"},
    "henry_hub": {"route": "natural-gas/pri/fut", "series": "RNGWHHD", "frequency": "daily", "fred": "DHHNGSP",
                  "label": "EIA Henry Hub natural gas spot ($/MMBtu)"},
    # petroleum/sum/sndw is the route EIA's own series-id lookup maps WCESTUS1 to (stoc/wstk tends to time out)
    "crude_stocks": {"route": "petroleum/sum/sndw", "series": "WCESTUS1", "frequency": "weekly", "fred": None,
                     "xls": "https://www.eia.gov/dnav/pet/hist_xls/WCESTUS1w.xls",
                     "label": "EIA U.S. commercial crude stocks ex-SPR (thousand bbl)"},
}


def validate_eia(payload):
    if (isinstance(payload, dict) and "error" not in payload and isinstance(payload.get("response"), dict)
            and "data" in payload["response"] and str(payload["response"].get("total", "")) != "0"):
        return
    detail = payload.get("error", payload) if isinstance(payload, dict) else payload
    raise PermanentAPIError(f"EIA: {str(detail)[:200]}")


def eia_from_api(spec):
    rows, offset = [], 0
    while True:  # 5,000 rows per page; `total` (a string) is the full match count
        params = {"api_key": API_KEYS["EIA_API_KEY"], "frequency": spec["frequency"], "data[0]": "value",
                  "facets[series][]": spec["series"], "start": MACRO_START,
                  "sort[0][column]": "period", "sort[0][direction]": "asc", "offset": offset, "length": EIA_PAGE_ROWS}
        payload = fetch(f"{EIA_URL}/{spec['route']}/data/", namespace="eia", params=params, literal_brackets=True,
                        ttl_hours=CACHE_TTL_HOURS["eia"], validate=validate_eia, min_interval_sec=0.5)
        batch = payload["response"]["data"]
        rows.extend(batch)
        offset += len(batch)
        if not batch or offset >= int(payload["response"].get("total") or 0):
            break
    if not rows:
        raise PermanentAPIError(f"EIA: 0 rows for {spec['series']} on {spec['route']}")
    df = pd.DataFrame(rows)
    s = pd.Series(pd.to_numeric(df["value"], errors="coerce").to_numpy(), index=pd.to_datetime(df["period"]),
                  name=spec["series"]).sort_index()
    return s[~s.index.duplicated(keep="last")]


def validate_xls(content):
    if not content[:8] == b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1":  # OLE2 signature of a real .xls workbook
        raise PermanentAPIError("EIA history file: not an .xls workbook")


def eia_from_history_file(spec):
    content = fetch(spec["xls"], namespace="eia_xls", fmt="bytes", ttl_hours=CACHE_TTL_HOURS["eia"],
                    validate=validate_xls)
    sheet = pd.read_excel(BytesIO(content), sheet_name="Data 1", skiprows=2)
    s = pd.Series(pd.to_numeric(sheet.iloc[:, 1], errors="coerce").to_numpy(),
                  index=pd.to_datetime(sheet.iloc[:, 0]), name=spec["series"]).dropna().sort_index()
    return s.loc[MACRO_START:]


def load_eia_series(name):
    spec = EIA_SERIES[name]
    if API_KEYS["EIA_API_KEY"]:
        try:
            s = eia_from_api(spec)
            log_source(spec["label"], "EIA API v2", s)
            return s
        except (PermanentAPIError, TransientAPIError) as exc:
            print(f"⚠️ EIA API failed for {spec['series']} ({exc}) → fallback")
    else:
        print(f"ℹ️ No EIA_API_KEY: {spec['series']} from its keyless fallback")
    if spec["fred"]:
        return load_fred_series(spec["fred"], label=f"{spec['label']} via FRED {spec['fred']}")
    s = eia_from_history_file(spec)
    log_source(spec["label"], "EIA history file (keyless)", s)
    return s

In [ ]:
# ### Cell 11 (Code)
BLS_URLS = {"v2": "https://api.bls.gov/publicAPI/v2/timeseries/data/",
            "v1": "https://api.bls.gov/publicAPI/v1/timeseries/data/"}
BLS_YEARS_PER_REQUEST = {"v2": 20, "v1": 10}
BLS_SERIES = {"payrolls": "CES0000000001", "unemployment": "LNS14000000", "wages": WAGE_SERIES}
BLS_FRED_MIRRORS = {"CES0000000001": "PAYEMS", "LNS14000000": "UNRATE",
                    "CES0500000003": "CES0500000003", "CES0500000008": "AHETPI"}


def validate_bls(payload):
    messages = payload.get("message", []) if isinstance(payload, dict) else [str(payload)]
    text = "; ".join(map(str, messages))
    if isinstance(payload, dict) and payload.get("status") == "REQUEST_SUCCEEDED":
        if "reduced" in text.lower():  # BLS silently drops the most recent years if a request spans too many
            raise PermanentAPIError(f"BLS truncated the year range: {text[:160]}")
        if not any(s.get("data") for s in payload.get("Results", {}).get("series", [])):
            raise PermanentAPIError(f"BLS: no data returned ({text[:160]})")
        return
    raise PermanentAPIError(f"BLS: {redact(text)[:220] or payload.get('status')}")  # e.g. daily threshold reached


def bls_from_api(series_ids, start_year, end_year):
    key = API_KEYS["BLS_API_KEY"]
    version = "v2" if key else "v1"
    span = BLS_YEARS_PER_REQUEST[version]
    records = []
    for y0 in range(start_year, end_year + 1, span):
        y1 = min(y0 + span - 1, end_year)
        body = {"seriesid": series_ids, "startyear": str(y0), "endyear": str(y1)}
        if key:
            body["registrationkey"] = key
        ttl = CACHE_TTL_HOURS["bls"] if y1 >= END_DATE.year - 1 else 24 * 30  # closed decades rarely change
        payload = fetch(BLS_URLS[version], namespace=f"bls_{version}", json_body=body, ttl_hours=ttl,
                        validate=validate_bls, min_interval_sec=1.0)
        for series in payload["Results"]["series"]:
            for item in series["data"]:
                if re.fullmatch(r"M(0[1-9]|1[0-2])", item["period"]):  # skips M13 = annual average
                    records.append({"series": series["seriesID"],
                                    "date": pd.Timestamp(int(item["year"]), int(item["period"][1:]), 1),
                                    "value": pd.to_numeric(item["value"], errors="coerce")})  # "-" = not collected
    wide = pd.DataFrame(records).pivot_table(index="date", columns="series", values="value", aggfunc="last")
    return wide.sort_index(), f"BLS API {version}" + ("" if key else " (keyless)")


def load_bls_panel():
    ids = list(BLS_SERIES.values())
    try:
        wide, source = bls_from_api(ids, int(MACRO_START[:4]), END_DATE.year)
        missing = [sid for sid in ids if sid not in wide]
        if missing:
            raise PermanentAPIError(f"no data returned for {missing}")
    except (PermanentAPIError, TransientAPIError) as exc:
        print(f"⚠️ BLS API unavailable ({exc}) → FRED mirrors of the same series")
        wide = pd.DataFrame({sid: load_fred_series(BLS_FRED_MIRRORS[sid], label=f"BLS {sid} via FRED") for sid in ids})
        source = "FRED mirror of BLS"
    panel = wide.rename(columns={v: k for k, v in BLS_SERIES.items()})[list(BLS_SERIES)]
    panel.index = month_end(panel.index)
    panel = panel.asfreq("ME")
    for col in panel:
        log_source(f"BLS {BLS_SERIES[col]} ({col})", source, panel[col],
                   note=f"{panel[col].isna().sum()} missing months" if panel[col].loc[panel[col].first_valid_index():].isna().any() else "")
    return panel

In [ ]:
# ### Cell 12 (Code)
RESULTS = {}  # key numbers from each module, reused by the findings and resume cells


def pct_change(x, periods=1):
    return x / x.shift(periods) - 1


def as_month_end(series):
    """Monthly series → regular month-end index (so .shift(k) means k months)."""
    s = series.dropna().copy()
    s.index = month_end(s.index)
    return s[~s.index.duplicated(keep="last")].asfreq("ME")


def point_in_time(series, lag_months, overrides=None, ffill_limit=2):
    """Re-stamp a reference-month series at the month-end when each value was first public, so `.loc[t]` is
    what an investor could know at the end of month t. `overrides` maps 'YYYY-MM' → actual release date."""
    s = series.dropna().copy()
    s.index = month_end(s.index)  # FRED/BLS stamp monthly data on the 1st; re-stamp so the lag counts from month-end
    available = s.index + pd.offsets.MonthEnd(lag_months)
    for ref, released in (overrides or {}).items():
        available = available.where(s.index != month_end([ref])[0], month_end([released])[0])
    known = pd.Series(s.to_numpy(), index=available).groupby(level=0).last()  # same release date → keep later month
    return known.asfreq("ME").ffill(limit=ffill_limit)


def drawdown(returns):
    equity = (1 + returns.fillna(0)).cumprod()
    peak = np.maximum.accumulate(np.maximum(equity.to_numpy(), 1.0))
    return pd.Series(equity.to_numpy() / peak - 1, index=returns.index)


def perf_stats(returns, rf=None, periods_per_year=TRADING_DAYS):
    r = returns.dropna()
    rf = rf.reindex(r.index).fillna(0.0) if rf is not None else 0.0
    years = len(r) / periods_per_year
    total = (1 + r).prod()
    excess = r - rf
    mdd = drawdown(r).min()
    cagr = total ** (1 / years) - 1
    return {"CAGR": cagr, "Volatility": r.std() * np.sqrt(periods_per_year),
            "Sharpe": excess.mean() / excess.std() * np.sqrt(periods_per_year) if excess.std() > 0 else np.nan,
            "Max drawdown": mdd, "Calmar": cagr / abs(mdd) if mdd < 0 else np.nan, "Total return": total - 1}


def expanding_robust_z(x, min_periods=ZSCORE_MIN_MONTHS, clip=3.0):
    """(x − median) / (IQR / 1.349), using only observations up to each date; robust to the 2020 outliers."""
    window = x.expanding(min_periods=min_periods)
    median, q75, q25 = window.median(), window.quantile(0.75), window.quantile(0.25)
    return ((x - median) / ((q75 - q25) / 1.349)).clip(-clip, clip)


def sahm_indicator(unrate):
    """3-month average unemployment minus its low over the prior 12 months (≥ 0.5 pp = Sahm-rule trigger)."""
    ma3 = unrate.rolling(3, min_periods=2).mean()
    # inputs have 0.1 pp precision, so exact 0.50 readings come out as 0.4999999…; round before any >= 0.5 test
    return (ma3 - ma3.shift(1).rolling(12, min_periods=6).min()).round(6)


def true_runs(mask):
    """[(first, last), ...] index labels of each consecutive run of True values."""
    mask = mask.fillna(False).astype(bool)
    run_id = mask.ne(mask.shift(fill_value=False)).cumsum()
    return [(g.index[0], g.index[-1]) for _, g in mask[mask].groupby(run_id[mask])]


def recession_periods(usrec):
    if usrec is None:
        return []
    monthly = as_month_end(usrec).fillna(0)
    return [(start - pd.offsets.MonthBegin(1), end) for start, end in true_runs(monthly == 1)]


def shade(fig, periods, color="rgba(128,128,128,0.18)", start=None, **subplot):
    for x0, x1 in periods:
        if start is None or x1 >= pd.Timestamp(start):
            fig.add_vrect(x0=x0, x1=x1, fillcolor=color, line_width=0, layer="below", **subplot)
    return fig


def add_legend_swatch(fig, name, color, **subplot):
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=name,
                             marker=dict(size=12, symbol="square", color=color)), **subplot)


def _number_formatter(pattern):
    return lambda v: pattern.format(v) if isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool) else v


def show_table(df, title=None, pct=(), dec2=(), dec0=(), dec3=(), sig=(), hide_index=False):
    patterns = {**{c: "{:.1%}" for c in pct}, **{c: "{:,.2f}" for c in dec2}, **{c: "{:,.0f}" for c in dec0},
                **{c: "{:.3f}" for c in dec3}, **{c: "{:,.6g}" for c in sig}}
    styler = df.style.format({k: _number_formatter(v) for k, v in patterns.items() if k in df.columns}, na_rep="—")
    if title:
        styler = styler.set_caption(title)
    if hide_index:
        styler = styler.hide(axis="index")
    display(styler)

In [ ]:
# ### Cell 13 (Code)
DATA_LOG.clear()


def safe_load(label, loader, *args):
    try:
        return loader(*args)
    except Exception as exc:  # one dead source shouldn't stop "Run all"
        print(f"❌ {label} failed: {redact(exc)[:200]} — modules that need it will be skipped")
        return None


prices = safe_load("Prices", load_all_prices, EQUITY_TICKERS) or {}
close = pd.DataFrame({t: df["close"] for t, df in prices.items()}).sort_index()
fred = {sid: safe_load(f"FRED {sid}", load_fred_series, sid) for sid in FRED_SERIES}
crypto_markets = safe_load("CoinGecko markets", load_coingecko_markets)
crypto_excluded = safe_load("CoinGecko categories", load_excluded_coin_ids) or {}
eia = {name: safe_load(f"EIA {name}", load_eia_series, name) for name in EIA_SERIES}
labor_raw = safe_load("BLS", load_bls_panel)

provenance = pd.DataFrame(DATA_LOG)
show_table(provenance, "Data provenance: which source each dataset actually came from", dec0=["rows"], hide_index=True)

USREC_PERIODS = recession_periods(fred.get("USREC"))
REC_COLOR = "rgba(128,128,128,0.18)"
print(f"{len(USREC_PERIODS)} NBER recessions since {MACRO_START[:4]}: "
      + ", ".join(f"{a:%Y-%m}→{b:%Y-%m}" for a, b in USREC_PERIODS))

### Cell 14 (Markdown)

## Module A — Trend following: 50/200-day moving-average crossover vs buy-and-hold SPY
**The idea.** Time-series momentum: assets that have been rising tend to keep rising over the next few months, and big bear markets unfold slowly enough for a trend filter to step aside. The classic "golden cross" holds SPY while its 50-day average is above its 200-day average and sits in cash otherwise. You expect to give up some return in choppy markets (whipsaws, late re-entry after V-shaped rebounds) in exchange for avoiding the deepest part of long drawdowns such as 2000–02 and 2008.

**Execution, no look-ahead.** The signal uses closes up to and including day *t*. The trade executes at the **open of day t+1**. The overnight gap from *t*'s close to *t+1*'s open still belongs to the old position, and only the intraday move belongs to the new one.

**Costs and cash.** Each change in position pays `COST_BPS` on the traded notional at the open. While out of the market the strategy earns the fed funds rate, the same rate used as the risk-free rate in the Sharpe ratio.

**Metrics.** CAGR, annualised volatility, Sharpe (excess over fed funds), maximum drawdown, Calmar (CAGR / |max DD|), time in market and trading frequency, plus a cost-sensitivity grid.

In [ ]:
# ### Cell 15 (Code)
def sma_crossover_signal(close_px, fast=FAST_MA, slow=SLOW_MA):
    """1 = long, 0 = flat, NaN while the slow average warms up. Uses closes up to day t only."""
    fast_ma, slow_ma = close_px.rolling(fast).mean(), close_px.rolling(slow).mean()
    return (fast_ma > slow_ma).astype(float).where(slow_ma.notna()), fast_ma, slow_ma


def backtest_next_open(px, signal, cost_bps, cash_rate):
    """Signal at close t → trade at open t+1. The overnight return accrues to the old weight, the intraday
    return to the new one; costs are charged on traded notional at the open; idle cash earns cash_rate."""
    weight = signal.shift(1)
    start = weight.first_valid_index()
    o, c, w = px["open"].loc[start:], px["close"].loc[start:], weight.loc[start:]
    w_prev = w.shift(1, fill_value=0.0)  # day one starts in cash
    rf = cash_rate.reindex(o.index).fillna(0.0)
    r_overnight = (o / c.shift(1) - 1).fillna(0.0)
    r_intraday = c / o - 1
    turnover = (w - w_prev).abs()
    growth = ((1 + w_prev * r_overnight + (1 - w_prev) * rf)
              * (1 - turnover * cost_bps / 1e4)
              * (1 + w * r_intraday))
    return pd.DataFrame({"strategy": growth - 1, "weight": w, "turnover": turnover})


def buy_and_hold_returns(px, start, cost_bps):
    """Buy at the open of `start` (the strategy's first tradable day, for a like-for-like window), then hold."""
    o, c = px["open"].loc[start:], px["close"].loc[start:]
    ret = c / c.shift(1) - 1
    ret.iloc[0] = c.iloc[0] / o.iloc[0] * (1 - cost_bps / 1e4) - 1
    return ret


def trade_log(bt):
    w = bt["weight"]
    entries = w.index[(w == 1) & (w.shift(1, fill_value=0) == 0)]
    exits = w.index[(w == 0) & (w.shift(1, fill_value=0) == 1)]
    rows = []
    for entry in entries:
        later = exits[exits > entry]
        exit_day = later[0] if len(later) else w.index[-1]
        window = bt["strategy"].loc[entry:exit_day]
        rows.append({"entry": entry, "exit": exit_day, "open": not len(later),
                     "return": (1 + window).prod() - 1, "days": len(window)})
    return pd.DataFrame(rows)


if "SPY" not in prices:
    print("⚠️ SPY prices unavailable — Module A skipped.")
else:
    spy = prices["SPY"]
    ff = fred.get("FEDFUNDS")
    rf_daily = ((ff.dropna() / 100 / TRADING_DAYS).reindex(spy.index, method="ffill").fillna(0.0)
                if ff is not None else pd.Series(0.0, index=spy.index))
    cash_rate = rf_daily if CASH_EARNS_RISK_FREE else rf_daily * 0.0

    signal, sma_fast, sma_slow = sma_crossover_signal(spy["close"])
    bt = backtest_next_open(spy, signal, COST_BPS, cash_rate)
    bt["buy_hold"] = buy_and_hold_returns(spy, bt.index[0], COST_BPS)
    trades = trade_log(bt)
    years = (bt.index[-1] - bt.index[0]).days / 365.25

    summary_A = pd.DataFrame({f"SMA {FAST_MA}/{SLOW_MA} crossover": perf_stats(bt["strategy"], rf_daily),
                              "Buy & hold SPY": perf_stats(bt["buy_hold"], rf_daily)}).T
    summary_A["Time in market"] = [bt["weight"].mean(), 1.0]
    summary_A["Trades / year"] = [bt["turnover"].gt(0).sum() / years, 1 / years]
    show_table(summary_A, f"SPY {bt.index[0]:%Y-%m-%d} → {bt.index[-1]:%Y-%m-%d} · {COST_BPS:g} bps per trade · "
                          "signal at close, fill at next open",
               pct=["CAGR", "Volatility", "Max drawdown", "Total return", "Time in market"],
               dec2=["Sharpe", "Calmar", "Trades / year"])

    closed = trades[~trades["open"]]
    print(f"Round trips: {len(closed)} closed (+{int(trades['open'].sum())} open) · win rate {(closed['return'] > 0).mean():.0%} · "
          f"median holding {closed['days'].median():.0f} trading days · best {closed['return'].max():+.1%} · worst {closed['return'].min():+.1%}")

    cost_rows = {}
    for bps in COST_GRID_BPS:
        strat = backtest_next_open(spy, signal, bps, cash_rate)["strategy"]
        s_stats = perf_stats(strat, rf_daily)
        b_stats = perf_stats(buy_and_hold_returns(spy, strat.index[0], bps), rf_daily)
        cost_rows[f"{bps:g} bps"] = {"Strategy CAGR": s_stats["CAGR"], "Strategy Sharpe": s_stats["Sharpe"],
                                     "CAGR minus buy & hold": s_stats["CAGR"] - b_stats["CAGR"]}
    show_table(pd.DataFrame(cost_rows).T, "Cost sensitivity (one-way cost per trade)",
               pct=["Strategy CAGR", "CAGR minus buy & hold"], dec2=["Strategy Sharpe"])

    RESULTS["A"] = {"start": bt.index[0], "end": bt.index[-1], "years": years, "cost_bps": COST_BPS,
                    "strategy": summary_A.iloc[0].to_dict(), "buy_hold": summary_A.iloc[1].to_dict(),
                    "round_trips": len(closed), "win_rate": (closed["return"] > 0).mean()}

In [ ]:
# ### Cell 16 (Code)
if "A" in RESULTS:
    equity = (1 + bt[["strategy", "buy_hold"]]).cumprod()
    cash_runs = true_runs(bt["weight"] == 0)

    fig = go.Figure()
    shade(fig, cash_runs, color="rgba(255,165,0,0.12)")
    add_legend_swatch(fig, "Strategy in cash", "rgba(255,165,0,0.35)")
    fig.add_trace(go.Scatter(x=equity.index, y=equity["buy_hold"], name="Buy & hold SPY", line=dict(color="#7f7f7f", width=1.6)))
    fig.add_trace(go.Scatter(x=equity.index, y=equity["strategy"], name=f"SMA {FAST_MA}/{SLOW_MA} crossover", line=dict(color="#1f77b4", width=2)))
    fig.update_xaxes(type="date")  # the legend-swatch trace has no x data, so plotly can't infer a date axis
    fig.update_yaxes(type="log", title="Growth of $1 (log scale)")
    fig.update_layout(title=f"Equity curve: SPY trend filter vs buy-and-hold ({COST_BPS:g} bps per trade)",
                      height=480, hovermode="x unified", legend=dict(orientation="h", y=1.08))
    fig.show()

    dd = pd.DataFrame({"Buy & hold SPY": drawdown(bt["buy_hold"]), f"SMA {FAST_MA}/{SLOW_MA}": drawdown(bt["strategy"])})
    fig = go.Figure()
    for name, color in zip(dd.columns, ["rgba(127,127,127,0.45)", "rgba(31,119,180,0.55)"]):
        fig.add_trace(go.Scatter(x=dd.index, y=dd[name], name=name, fill="tozeroy", line=dict(width=1, color=color)))
    fig.update_yaxes(title="Drawdown from previous peak", tickformat=".0%")
    fig.update_layout(title="Drawdowns", height=380, hovermode="x unified", legend=dict(orientation="h", y=1.1))
    fig.show()

    yearly = (1 + bt[["strategy", "buy_hold"]]).groupby(bt.index.year).prod() - 1
    gap = yearly["strategy"] - yearly["buy_hold"]
    fig = go.Figure(go.Bar(x=gap.index, y=gap, marker_color=np.where(gap >= 0, "#2ca02c", "#d62728"),
                           hovertemplate="%{x}: %{y:+.1%}<extra></extra>"))
    fig.update_yaxes(title="Strategy minus buy & hold", tickformat=".0%")
    fig.update_layout(title="Calendar-year return difference (partial first/last years)", height=340)
    fig.show()

### Cell 17 (Markdown)

## Module B — Macro dashboard: the yield curve, inflation and unemployment
**Yield curve.** The 10-year minus 2-year Treasury spread compares long rates (expected future short rates plus a term premium) with rates tied closely to Fed policy. When it inverts, markets expect cuts, usually because tight policy is expected to slow the economy, and banks that borrow short and lend long see margins squeezed. Every U.S. recession since the late 1970s was preceded by at least a brief daily inversion, usually a sustained one, with long and variable lags of roughly 1–2 years. The 2022–24 inversion, the longest on record, had not been followed by an NBER recession as of this data.

**Recession shading.** Grey bands are NBER recessions from FRED's `USREC`. NBER announces dates many months after the fact, so the bands are hindsight and are never used as an input.

**Inflation and labor.** CPI year-over-year is compared with the Fed's 2% goal (the Fed formally targets PCE, which typically runs a bit below CPI). Unemployment is shown with the **Sahm rule**: a recession signal when the 3-month average jobless rate rises 0.5 pp above its low of the prior 12 months.

**Inversion flag.** Daily flag = spread below zero. A *sustained* episode is a run of months with a negative monthly-average spread, at least two months long, so blips don't count. Runs separated by at most three positive months are merged, but never across the start of a recession. Each recession is credited to the first episode that began within 36 months before it, so every recession gets exactly one lead time.

In [ ]:
# ### Cell 18 (Code)
def months_between(a, b):
    return (b.to_period("M") - a.to_period("M")).n


def inversion_episodes(spread_m, usrec_m, merge_gap_months=3, min_inverted_months=2, max_lead_months=36):
    rec_starts = usrec_m.index[(usrec_m == 1) & (usrec_m.shift(1, fill_value=0) == 0)]
    episodes = []
    for month in spread_m.index[spread_m < 0]:
        if episodes:
            gap = months_between(episodes[-1][1], month) - 1
            crosses_recession = ((rec_starts > episodes[-1][1]) & (rec_starts <= month)).any()
            if gap <= merge_gap_months and not crosses_recession:
                episodes[-1][1] = month
                continue
        episodes.append([month, month])
    episodes = [(a, b) for a, b in episodes if (spread_m.loc[a:b] < 0).sum() >= min_inverted_months]

    rows, credited = [], set()
    for first, last in episodes:
        window = spread_m.loc[first:last]
        inside = usrec_m.get(first, 0) == 1  # the inversion carried on into a recession that had already started
        nxt = rec_starts[(rec_starts > first) & (rec_starts <= first + pd.DateOffset(months=max_lead_months))]
        censored = first == spread_m.index[0]  # already inverted when the data begin: true start unknown
        lead = months_between(first, nxt[0]) if len(nxt) and not inside else np.nan
        counts = bool(len(nxt)) and not (censored or inside) and nxt[0] not in credited
        if counts:
            credited.add(nxt[0])
        next_label = "(continues during a recession)" if inside else f"{nxt[0]:%Y-%m}" if len(nxt) else "none (yet)"
        rows.append({"first inverted month": f"{first:%Y-%m}" + (" (sample start)" if censored else ""),
                     "last inverted month": f"{last:%Y-%m}",
                     "inverted months": int((window < 0).sum()), "deepest monthly avg (pp)": window.min(),
                     "next NBER recession (≤ 36m)": next_label,
                     "lead (months)": lead, "in lead stats": "yes" if counts else "no"})
    return pd.DataFrame(rows)


def snapshot_row(name, s, unit, signal=""):
    s = s.dropna()
    last = s.index[-1]

    def months_ago(k):
        earlier = s.loc[:last - pd.DateOffset(months=k)]
        return earlier.iloc[-1] if len(earlier) else np.nan

    return {"indicator": name, "latest": s.iloc[-1], "as of": f"{last:%Y-%m-%d}", "3m ago": months_ago(3),
            "12m ago": months_ago(12), "unit": unit, "signal": signal}


MODULE_B_SERIES = ["T10Y2Y", "DGS10", "DGS2", "CPIAUCSL", "UNRATE", "FEDFUNDS"]
if any(fred.get(s) is None for s in MODULE_B_SERIES):
    print(f"⚠️ FRED series missing ({[s for s in MODULE_B_SERIES if fred.get(s) is None]}) — Module B skipped.")
else:
    spread_d = fred["T10Y2Y"].dropna()
    y10, y2 = fred["DGS10"].dropna(), fred["DGS2"].dropna()
    spread_m = spread_d.resample("ME").mean()
    cpi_yoy = pct_change(as_month_end(fred["CPIAUCSL"]), 12) * 100
    unrate = as_month_end(fred["UNRATE"])
    fedfunds = as_month_end(fred["FEDFUNDS"])
    usrec_m = (as_month_end(fred["USREC"]).fillna(0) if fred.get("USREC") is not None
               else pd.Series(0.0, index=spread_m.index))
    sahm = sahm_indicator(unrate)

    inverted_d = spread_d < 0
    flips = inverted_d.index[inverted_d != inverted_d.iloc[-1]]
    days_in_state = int((inverted_d.index > flips.max()).sum()) if len(flips) else len(inverted_d)
    share_inverted_1y = inverted_d.loc[spread_d.index[-1] - pd.DateOffset(years=1):].mean()
    state = "INVERTED" if inverted_d.iloc[-1] else "positive"
    ff_12m = fedfunds.diff(12).dropna().iloc[-1]

    snapshot = pd.DataFrame([
        snapshot_row("10Y Treasury", y10, "%"),
        snapshot_row("2Y Treasury", y2, "%"),
        snapshot_row("10Y − 2Y spread", spread_d, "pp", f"{state} for {days_in_state} trading days; "
                                                          f"inverted {share_inverted_1y:.0%} of last 12m"),
        snapshot_row("CPI inflation (YoY)", cpi_yoy, "%", "above 3%" if cpi_yoy.dropna().iloc[-1] > 3 else
                     ("near 2% goal" if cpi_yoy.dropna().iloc[-1] >= 1.5 else "below goal")),
        snapshot_row("Unemployment rate", unrate, "%"),
        snapshot_row("Sahm-rule indicator", sahm, "pp", "TRIGGERED (≥ 0.5)" if sahm.dropna().iloc[-1] >= 0.5 else "not triggered"),
        snapshot_row("Fed funds (monthly avg)", fedfunds, "%", f"{ff_12m:+.2f} pp over 12m ({'easing' if ff_12m < 0 else 'tightening' if ff_12m > 0 else 'unchanged'})"),
    ])
    show_table(snapshot, "Macro snapshot (latest published values)", dec2=["latest", "3m ago", "12m ago"], hide_index=True)

    episodes = inversion_episodes(spread_m, usrec_m)
    show_table(episodes, "Sustained inversions (monthly-average 10Y−2Y < 0 for 2+ months) and the next NBER recession",
               dec2=["deepest monthly avg (pp)"], dec0=["lead (months)"], hide_index=True)
    leads = episodes.loc[episodes["in lead stats"] == "yes", "lead (months)"]
    if len(leads):
        print(f"Lead from first inverted month to recession start, one per recession: median {leads.median():.0f} months "
              f"(range {leads.min():.0f}–{leads.max():.0f}, n={len(leads)} recessions)")

    RESULTS["B"] = {"spread": spread_d.iloc[-1], "spread_date": spread_d.index[-1], "inverted": bool(inverted_d.iloc[-1]),
                    "cpi_yoy": cpi_yoy.dropna().iloc[-1], "unrate": unrate.dropna().iloc[-1],
                    "sahm": sahm.dropna().iloc[-1], "median_lead": leads.median(), "n_episodes": len(episodes)}

In [ ]:
# ### Cell 19 (Code)
if "B" in RESULTS:
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=["Treasury yields (%)", "10Y − 2Y spread (pp) · red = inverted",
                                        "CPI inflation, year over year (%)", "Unemployment rate (%) · ◆ = Sahm rule triggered"])
    fig.add_trace(go.Scatter(x=y10.index, y=y10, name="10Y", line=dict(color="#1f77b4", width=1.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=y2.index, y=y2, name="2Y", line=dict(color="#ff7f0e", width=1.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=spread_d.index, y=spread_d.clip(lower=0), name="spread ≥ 0", fill="tozeroy",
                             line=dict(color="rgba(44,160,44,0.9)", width=0.8)), row=2, col=1)
    fig.add_trace(go.Scatter(x=spread_d.index, y=spread_d.clip(upper=0), name="inverted", fill="tozeroy",
                             line=dict(color="rgba(214,39,40,0.9)", width=0.8)), row=2, col=1)
    fig.add_trace(go.Scatter(x=cpi_yoy.index, y=cpi_yoy, name="CPI YoY", line=dict(color="#9467bd")), row=3, col=1)
    fig.add_hline(y=2, line_dash="dot", line_color="black", annotation_text="2%", row=3, col=1)
    fig.add_trace(go.Scatter(x=unrate.index, y=unrate, name="Unemployment", line=dict(color="#8c564b")), row=4, col=1)
    triggered = unrate[sahm >= 0.5]
    fig.add_trace(go.Scatter(x=triggered.index, y=triggered, mode="markers", name="Sahm ≥ 0.5",
                             marker=dict(symbol="diamond", size=6, color="#d62728")), row=4, col=1)
    shade(fig, USREC_PERIODS, color=REC_COLOR, row="all", col=1)
    add_legend_swatch(fig, "NBER recession", "rgba(128,128,128,0.45)", row=1, col=1)
    fig.update_layout(height=1050, title="FRED macro dashboard (grey = NBER recessions)", hovermode="x unified",
                      legend=dict(orientation="h", yanchor="top", y=-0.04))
    fig.show()

### Cell 20 (Markdown)

## Module C — Crypto screener: momentum × turnover
**The idea.** Cross-sectional momentum (recent winners keep outperforming for a while) is one of the most persistent effects in crypto, but it is strongest, and safest to trade, where there is real liquidity. The screener ranks the top-100 coins on:
- **Momentum:** a blend of the 7-day and 24-hour percentage change (7-day weighted more heavily, since 24-hour moves are mostly noise and short-term reversal).
- **Turnover:** 24-hour volume / market cap, a proxy for liquidity and attention. Very high turnover can also flag wash trading or a pump, so read it with the momentum column.

Both are converted to **percentile ranks** within the filtered universe, so scores don't depend on units or outliers. By default the screen excludes stablecoins, tokenized funds and gold tokens, and any wrapped or staked token that slips through. They track another asset (USD, T-bills, gold, BTC/ETH) rather than crypto risk, so they would only add noise. (Since February 2026 CoinGecko leaves wrapped and liquid-staked tokens out of `/coins/markets` by default.)

**Caveat.** This is a point-in-time snapshot ranking, not a backtest. CoinGecko's free tier doesn't give point-in-time historical top-100 lists, so testing it properly would need survivorship-free history. Edit the filters at the top of the next cell (Colab shows them as a form).

In [ ]:
# ### Cell 21 (Code)
MIN_MARKET_CAP_USD = 1000000000      # @param {type:"number"}
MIN_VOLUME_24H_USD = 20000000        # @param {type:"number"}
EXCLUDE_STABLECOINS = True           # @param {type:"boolean"}
EXCLUDE_TOKENIZED_AND_WRAPPED = True # @param {type:"boolean"}
MOMENTUM_WEIGHT = 0.6                # @param {type:"slider", min:0, max:1, step:0.05}
SHARE_OF_7D_IN_MOMENTUM = 0.7        # @param {type:"slider", min:0, max:1, step:0.05}
TOP_N = 15                           # @param {type:"integer"}


def exclusion_reason(row):
    tagged = crypto_excluded.get(row["id"])
    if tagged == "stablecoin" and EXCLUDE_STABLECOINS:
        return "stablecoin"
    if tagged in ("tokenized fund / gold", "wrapped / staked") and EXCLUDE_TOKENIZED_AND_WRAPPED:
        return tagged
    if not row["market_cap"] >= MIN_MARKET_CAP_USD:
        return "market cap below minimum"
    if not row["total_volume"] >= MIN_VOLUME_24H_USD:
        return "volume below minimum"
    if pd.isna(row["chg_7d"]) or pd.isna(row["chg_24h"]):
        return "missing % change"
    return ""


def screen_crypto(markets):
    df = markets.copy()
    df["turnover"] = df["total_volume"] / df["market_cap"].where(df["market_cap"] > 0)
    df["excluded"] = df.apply(exclusion_reason, axis=1)
    universe = df[df["excluded"] == ""].copy()
    universe["momentum_score"] = (SHARE_OF_7D_IN_MOMENTUM * universe["chg_7d"].rank(pct=True)
                                  + (1 - SHARE_OF_7D_IN_MOMENTUM) * universe["chg_24h"].rank(pct=True))
    universe["turnover_score"] = universe["turnover"].rank(pct=True)
    universe["composite"] = MOMENTUM_WEIGHT * universe["momentum_score"] + (1 - MOMENTUM_WEIGHT) * universe["turnover_score"]
    universe = universe.sort_values("composite", ascending=False)
    universe.insert(0, "screen_rank", range(1, len(universe) + 1))
    return universe, df.loc[df["excluded"] != "", "excluded"].value_counts()


if crypto_markets is None or crypto_markets.empty:
    print("⚠️ CoinGecko data unavailable — Module C skipped.")
else:
    ranked, excluded_counts = screen_crypto(crypto_markets)
    print(f"Snapshot {crypto_markets['last_updated'].max()} · {len(crypto_markets)} coins → {len(ranked)} pass the filters. "
          "Excluded: " + ", ".join(f"{k} {v}" for k, v in excluded_counts.items()))
    table = ranked.head(TOP_N).assign(mcap_bn=lambda d: d["market_cap"] / 1e9, volume_mn=lambda d: d["total_volume"] / 1e6,
                                      chg_24h=lambda d: d["chg_24h"] / 100, chg_7d=lambda d: d["chg_7d"] / 100,
                                      current_price=lambda d: d["current_price"].map(
                                          lambda p: f"{p:,.2f}" if p >= 1 else f"{p:.4f}" if p >= 0.01 else f"{p:.10f}".rstrip("0")))
    table = table[["screen_rank", "name", "symbol", "market_cap_rank", "current_price", "mcap_bn", "volume_mn",
                   "chg_24h", "chg_7d", "turnover", "momentum_score", "turnover_score", "composite"]]
    table.columns = ["#", "name", "symbol", "mcap rank", "price ($)", "mcap ($bn)", "24h vol ($mn)", "24h %", "7d %",
                     "vol / mcap", "momentum pct", "turnover pct", "composite"]
    show_table(table, f"Top {TOP_N} by composite score (momentum weight {MOMENTUM_WEIGHT:.0%})", hide_index=True,
               pct=["24h %", "7d %", "vol / mcap"], dec2=["mcap ($bn)", "momentum pct", "turnover pct", "composite"],
               dec0=["24h vol ($mn)"])

    fig = go.Figure(go.Scatter(
        x=ranked["chg_7d"], y=ranked["turnover"], mode="markers+text",
        text=np.where(ranked["screen_rank"] <= TOP_N, ranked["symbol"], ""), textposition="top center",
        marker=dict(size=np.clip(np.sqrt(ranked["market_cap"] / 1e9) * 4, 6, 60), color=ranked["composite"],
                    colorscale="Viridis", showscale=True, colorbar=dict(title="composite"), line=dict(width=0.5, color="white")),
        customdata=np.stack([ranked["name"], ranked["market_cap"] / 1e9, ranked["chg_24h"]], axis=1),
        hovertemplate="%{customdata[0]}<br>7d %{x:.1f}% · 24h %{customdata[2]:.1f}%<br>vol/mcap %{y:.2%}<br>mcap $%{customdata[1]:.1f}bn<extra></extra>"))
    fig.add_vline(x=0, line_dash="dot", line_color="grey")
    fig.update_xaxes(title="7-day price change (%)")
    fig.update_yaxes(title="24h volume / market cap (log)", type="log", tickformat=".1%")
    fig.update_layout(title="Momentum vs turnover (bubble size ∝ √market cap)", height=520)
    fig.show()

    top = ranked.head(TOP_N).iloc[::-1]
    fig = go.Figure()
    fig.add_trace(go.Bar(y=top["symbol"], x=MOMENTUM_WEIGHT * top["momentum_score"], orientation="h", name="momentum contribution", marker_color="#1f77b4"))
    fig.add_trace(go.Bar(y=top["symbol"], x=(1 - MOMENTUM_WEIGHT) * top["turnover_score"], orientation="h", name="turnover contribution", marker_color="#ff7f0e"))
    fig.update_layout(barmode="stack", title=f"Top {TOP_N}: composite score decomposition", height=460,
                      xaxis_title="composite score (0–1)", legend=dict(orientation="h", y=1.08))
    fig.show()

    leader = ranked.iloc[0]
    RESULTS["C"] = {"n_universe": len(ranked), "leader": f"{leader['name']} ({leader['symbol']})",
                    "leader_7d": leader["chg_7d"], "snapshot": crypto_markets["last_updated"].max()}

### Cell 22 (Markdown)

## Module D — Energy: commodity exposure of XLE/XOM and the weekly inventory report
**Rolling correlations.** Energy equities are claims on future oil and gas cash flows, so their returns should co-move with commodity prices, but the link changes with the cycle, hedging, capital discipline and how much of the sector is gas-weighted. Rolling 3-month and 1-year correlations of **daily log returns** show how stable that exposure is. Prices are inner-joined on common trading days, and the one negative WTI print (−$36.98 on 2020-04-20) is dropped because log returns need positive prices. Weekly correlations are reported as well: the EIA spot assessment and the 4 pm equity close aren't simultaneous, which pulls daily correlations down.

**Inventory test.** Every Wednesday at 10:30 ET the EIA's Weekly Petroleum Status Report publishes crude stocks for the week ending the previous Friday. A larger-than-normal build signals weak demand or excess supply, which is bearish for oil. If markets are efficient, the price reaction should happen on release day, and the surprise should **not** predict the *next* week's return.
- **Surprise** = weekly change in stocks minus the average change for the same week of the year over the *previous five years* (a seasonal norm built from past data only; consensus forecasts aren't freely available).
- **Timing**: the release date is the Wednesday after the week-ending Friday, moved to Thursday when a federal holiday falls Monday–Wednesday, plus a hand-coded table of irregular delays (Christmas 2025). The forward return starts at the close of the first trading day **after** the release day, which leaves a one-session margin if a release slipped further than the rules know, and runs 5 trading days.
- **Tests**: OLS with Newey-West (HAC) errors, Spearman rank IC, surprise-quintile averages, split-sample stability, and a separate release-day reaction as a sanity check that the signal moves prices at all.

In [ ]:
# ### Cell 23 (Code)
def aligned_energy_returns():
    levels = pd.concat({"WTI": eia["wti"], "Henry Hub": eia["henry_hub"], "XLE": close["XLE"], "XOM": close["XOM"]},
                       axis=1, sort=True).loc[ENERGY_CORR_START:].dropna()
    levels = levels.where(levels > 0)  # WTI settled at −$36.98 on 2020-04-20; log returns need positive prices
    return np.log(levels).diff().dropna(), levels


ENERGY_PAIRS = [("WTI", "XLE"), ("WTI", "XOM"), ("Henry Hub", "XLE"), ("Henry Hub", "XOM")]

if eia.get("wti") is None or eia.get("henry_hub") is None or not {"XLE", "XOM"} <= set(close.columns):
    print("⚠️ Energy prices unavailable — rolling-correlation study skipped.")
else:
    energy_rets, energy_levels = aligned_energy_returns()
    weekly_rets = np.log(energy_levels.resample("W-FRI").last()).diff().dropna()
    rolling_corr = {w: pd.DataFrame({f"{a} ↔ {b}": energy_rets[a].rolling(w).corr(energy_rets[b]) for a, b in ENERGY_PAIRS})
                    for w in CORR_WINDOWS}

    rows = []
    for a, b in ENERGY_PAIRS:
        label = f"{a} ↔ {b}"
        beta = energy_rets[b].cov(energy_rets[a]) / energy_rets[a].var()
        long_window = rolling_corr[max(CORR_WINDOWS)][label].dropna()
        rows.append({"pair": label, "daily corr (full)": energy_rets[a].corr(energy_rets[b]),
                     "weekly corr (full)": weekly_rets[a].corr(weekly_rets[b]),
                     f"latest {min(CORR_WINDOWS)}d": rolling_corr[min(CORR_WINDOWS)][label].dropna().iloc[-1],
                     f"latest {max(CORR_WINDOWS)}d": long_window.iloc[-1],
                     f"{max(CORR_WINDOWS)}d min": long_window.min(), f"{max(CORR_WINDOWS)}d max": long_window.max(),
                     "daily beta to commodity": beta})
    corr_table = pd.DataFrame(rows)
    show_table(corr_table, f"Energy equities vs commodities · daily log returns {energy_rets.index[0]:%Y-%m-%d} → {energy_rets.index[-1]:%Y-%m-%d}",
               dec2=[c for c in corr_table.columns if c != "pair"], hide_index=True)

    fig = make_subplots(rows=len(CORR_WINDOWS), cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=[f"{w}-day rolling correlation of daily log returns" for w in CORR_WINDOWS])
    colors = {"WTI ↔ XLE": "#1f77b4", "WTI ↔ XOM": "#aec7e8", "Henry Hub ↔ XLE": "#d62728", "Henry Hub ↔ XOM": "#ff9896"}
    for i, w in enumerate(CORR_WINDOWS, start=1):
        for col in rolling_corr[w]:
            fig.add_trace(go.Scatter(x=rolling_corr[w].index, y=rolling_corr[w][col], name=col, legendgroup=col,
                                     showlegend=i == 1, line=dict(color=colors[col], width=1.3)), row=i, col=1)
        fig.add_hline(y=0, line_color="black", line_width=0.6, row=i, col=1)
    shade(fig, USREC_PERIODS, color=REC_COLOR, start=energy_rets.index[0], row="all", col=1)
    fig.update_yaxes(range=[-0.4, 1], title="correlation")
    fig.update_layout(height=640, title="How tightly do energy stocks track oil and gas?", hovermode="x unified",
                      legend=dict(orientation="h", yanchor="top", y=-0.08))
    fig.show()

    RESULTS["D_corr"] = {"wti_xle": corr_table.set_index("pair").loc["WTI ↔ XLE", "daily corr (full)"],
                         "wti_xle_weekly": corr_table.set_index("pair").loc["WTI ↔ XLE", "weekly corr (full)"],
                         "hh_xle": corr_table.set_index("pair").loc["Henry Hub ↔ XLE", "daily corr (full)"]}

In [ ]:
# ### Cell 24 (Code)
def wpsr_release_dates(week_ending):
    """Wednesday after the week-ending Friday, or Thursday when a federal holiday falls Mon–Wed of that week."""
    week_ending = pd.DatetimeIndex(week_ending)
    wednesday = week_ending + pd.Timedelta(days=5)
    holidays = USFederalHolidayCalendar().holidays(start=wednesday.min() - pd.Timedelta(days=7), end=wednesday.max())
    hits = holidays.searchsorted(wednesday, side="right") - holidays.searchsorted(wednesday - pd.Timedelta(days=2), side="left")
    release = wednesday + pd.to_timedelta((hits > 0).astype(int), unit="D")
    overrides = {pd.Timestamp(k): pd.Timestamp(v) for k, v in WPSR_RELEASE_OVERRIDES.items()}
    return pd.DatetimeIndex([overrides.get(we, rel) for we, rel in zip(week_ending, release)])


def inventory_events(stocks_kbbl, closes):
    ev = (stocks_kbbl.dropna() / 1000).rename("stocks_mmbbl").to_frame()
    ev["change"] = ev["stocks_mmbbl"].diff()
    iso = ev.index.isocalendar()
    ev["year"], ev["week"] = iso["year"].astype(int).to_numpy(), iso["week"].astype(int).clip(upper=52).to_numpy()
    by_week = ev.pivot_table(index="year", columns="week", values="change", aggfunc="mean")
    norm = by_week.shift(1).rolling(5, min_periods=3).mean()  # row y = mean of years y-5 … y-1 only
    ev["seasonal_norm"] = [norm.at[y, w] if (y in norm.index and w in norm.columns) else np.nan
                           for y, w in zip(ev["year"], ev["week"])]
    ev["surprise"] = ev["change"] - ev["seasonal_norm"]
    ev["release"] = wpsr_release_dates(ev.index)

    days = closes.index
    react = days.searchsorted(ev["release"])                          # first session on/after release day
    entry = days.searchsorted(ev["release"] + pd.Timedelta(days=1))   # first session strictly after release day
    exit_ = entry + INVENTORY_HOLD_DAYS
    ok = (ev["release"] > days[0]).to_numpy() & (exit_ < len(days)) & (react >= 1)
    ev = ev[ok].copy()
    react, entry, exit_ = react[ok], entry[ok], exit_[ok]
    ev["entry_date"] = days[entry]
    for tkr in closes:
        px_ = closes[tkr].to_numpy()
        ev[f"{tkr} next week"] = px_[exit_] / px_[entry] - 1
        ev[f"{tkr} release day"] = px_[react] / px_[react - 1] - 1
    for tkr in ("XLE", "XOM"):
        ev[f"{tkr} − SPY next week"] = ev[f"{tkr} next week"] - ev["SPY next week"]
    return ev.dropna(subset=["surprise"])


def predictive_test(ev, target, x="surprise", hac_lags=2):
    data = ev[[target, x]].dropna()
    fit = sm.OLS(data[target], sm.add_constant(data[x])).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    quint = pd.qcut(data[x], 5, labels=False)
    q_means = data.groupby(quint)[target].mean()
    ic, ic_p = stats.spearmanr(data[x], data[target])
    sd = data[x].std()
    return {"target": target, "signal": x, "N": len(data), "β per 1σ of signal (bps)": fit.params[x] * sd * 1e4,
            "t (HAC)": fit.tvalues[x], "p-value": fit.pvalues[x], "R²": fit.rsquared, "Spearman IC": ic,
            "IC p-value": ic_p, "Q5 − Q1 (bps)": (q_means.iloc[-1] - q_means.iloc[0]) * 1e4}


if eia.get("crude_stocks") is None or not {"XLE", "XOM", "SPY"} <= set(close.columns):
    print("⚠️ Inventory data or prices unavailable — inventory test skipped.")
else:
    events = inventory_events(eia["crude_stocks"], close[["XLE", "XOM", "SPY"]].dropna())
    early, late = events.iloc[: len(events) // 2], events.iloc[len(events) // 2:]
    targets = ["XLE next week", "XOM next week", "XLE − SPY next week", "XOM − SPY next week"]
    tests = [predictive_test(events, t) for t in targets]
    tests.insert(1, {**predictive_test(events, "XLE next week", x="change"), "target": "XLE next week"})
    tests += [{**predictive_test(part, "XLE next week"), "target": f"XLE next week ({part.index[0]:%Y-%m} → {part.index[-1]:%Y-%m})"}
              for part in (early, late)]
    tests += [{**predictive_test(events, f"{t} release day"), "target": f"{t} release day (contemporaneous)"} for t in ("XLE", "XOM")]
    inv_table = pd.DataFrame(tests)
    show_table(inv_table, f"Do crude-inventory changes predict energy returns? {len(events)} weekly reports, "
                          f"{events.index[0]:%Y-%m} → {events.index[-1]:%Y-%m}",
               dec2=["β per 1σ of signal (bps)", "t (HAC)", "p-value", "Spearman IC", "IC p-value", "Q5 − Q1 (bps)"],
               dec3=["R²"], dec0=["N"], hide_index=True)
    print(f"signal = 'surprise' (weekly change minus its 5-year seasonal norm, σ = {events['surprise'].std():.2f} mn bbl) or the "
          f"raw weekly 'change' (σ = {events['change'].std():.2f} mn bbl). Positive = bigger build (bearish oil). "
          "Release-day rows are contemporaneous and NOT tradable; they only check that the market reacts.")

    quint = pd.qcut(events["surprise"], 5, labels=["Q1 big draw", "Q2", "Q3", "Q4", "Q5 big build"])
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Next-week return (entry after release)",
                                                        "Release-day return (contemporaneous)"])
    for i, horizon in enumerate(["next week", "release day"], start=1):
        for tkr, color in [("XLE", "#1f77b4"), ("XOM", "#ff7f0e")]:
            means = events.groupby(quint, observed=True)[f"{tkr} {horizon}"].mean() * 1e4
            fig.add_trace(go.Bar(x=means.index.astype(str), y=means, name=tkr, marker_color=color, showlegend=i == 1), row=1, col=i)
    fig.update_yaxes(title="mean return (bps)", row=1, col=1)
    fig.update_layout(barmode="group", height=440, legend=dict(orientation="h", yanchor="top", y=-0.15),
                      title="Energy-stock returns by surprise quintile (quintile edges use the full sample: descriptive, not a trading rule)")
    fig.show()

    row = inv_table[inv_table["signal"] == "surprise"].set_index("target")
    RESULTS["D_inv"] = {"n": len(events), "xle_t": row.loc["XLE next week", "t (HAC)"], "xle_p": row.loc["XLE next week", "p-value"],
                        "xle_ic": row.loc["XLE next week", "Spearman IC"],
                        "xle_react_t": row.loc["XLE release day (contemporaneous)", "t (HAC)"],
                        "xle_react_beta": row.loc["XLE release day (contemporaneous)", "β per 1σ of signal (bps)"]}

### Cell 25 (Markdown)

## Module E — A composite labor-market score from BLS data
**The idea.** No single labor statistic is reliable on its own. Payrolls get revised, the unemployment rate lags, and wages respond slowly. Averaging standardised signals cuts noise. The score combines:

| Component | Transformation | Sign |
|---|---|---|
| Payroll growth | 3-month average monthly change in nonfarm payrolls (thousands) | + |
| Unemployment change | 3-month change in the unemployment rate (pp) | − (rising joblessness = weaker) |
| Wage growth | year-over-year % change in average hourly earnings (production & nonsupervisory, `CES0500000008`, which runs back to 1964; set `WAGE_SERIES = "CES0500000003"` for all employees, from 2006) | + (firms bidding for workers) |

**Composition guard.** When payrolls collapse, *average* hourly earnings jump because low-wage jobs disappear first: the series rose about 8% year over year in April 2020. That is not a tight labor market, so while the payroll z-score is −2 or lower, the wage z-score is capped at 0.

**Standardising without look-ahead.** Each component is turned into an **expanding robust z-score**, `(x − median) / (IQR / 1.349)`, computed only from data available up to that month (36-month warm-up) and capped at ±3. Medians and IQRs are used instead of means and standard deviations so the 2020 shock (−20 million jobs in one month) doesn't distort every later reading. The score is the average of the available components (at least 2 of 3).

**Release timing.** The score for month *m* uses data published in early *m+1*. Module F shifts it by one month before using it.

In [ ]:
# ### Cell 26 (Code)
if labor_raw is None:
    print("⚠️ BLS data unavailable — Module E skipped.")
else:
    labor_features = pd.DataFrame({
        "payroll_3m_avg_chg": labor_raw["payrolls"].diff().rolling(3).mean(),
        "unemp_3m_chg": labor_raw["unemployment"].diff(3),
        "wage_yoy": pct_change(labor_raw["wages"], 12) * 100,
    })
    labor_z = pd.DataFrame({
        "Payroll growth": expanding_robust_z(labor_features["payroll_3m_avg_chg"]),
        "Unemployment change (inverted)": -expanding_robust_z(labor_features["unemp_3m_chg"]),
        "Wage growth": expanding_robust_z(labor_features["wage_yoy"]),
    })
    # Composition guard: when payrolls collapse, *average* wages jump because low-wage jobs vanish (AHE +8% YoY in
    # Apr 2020), which is not a tight labor market. Cap the wage z at 0 while the payroll z is deeply negative.
    collapsing = labor_z["Payroll growth"] <= -2
    labor_z.loc[collapsing, "Wage growth"] = labor_z.loc[collapsing, "Wage growth"].clip(upper=0)
    labor_score = labor_z.mean(axis=1).where(labor_z.notna().sum(axis=1) >= 2).rename("labor_score")
    valid = labor_score.dropna()
    latest_pct = (valid <= valid.iloc[-1]).mean()

    fig = go.Figure()
    for col, color in zip(labor_z.columns, ["#1f77b4", "#ff7f0e", "#2ca02c"]):
        fig.add_trace(go.Scatter(x=labor_z.index, y=labor_z[col], name=col, line=dict(color=color, width=1, dash="dot"), opacity=0.7))
    fig.add_trace(go.Scatter(x=labor_score.index, y=labor_score, name="Composite labor score", line=dict(color="black", width=2.2)))
    fig.add_hline(y=0, line_color="grey", line_width=0.8)
    shade(fig, USREC_PERIODS, color=REC_COLOR, start=valid.index[0])
    add_legend_swatch(fig, "NBER recession", "rgba(128,128,128,0.45)")
    fig.update_xaxes(range=[valid.index[0], valid.index[-1]])
    fig.update_yaxes(title="robust z-score (capped ±3)")
    fig.update_layout(title=f"BLS composite labor-market score (latest {valid.iloc[-1]:+.2f}, "
                            f"{latest_pct * 100:.0f}th percentile of history; reference month {valid.index[-1]:%Y-%m})",
                      height=480, hovermode="x unified", legend=dict(orientation="h", y=1.1))
    fig.show()

    recent = pd.concat([labor_raw, labor_features, labor_z, labor_score], axis=1, sort=True).dropna(how="all").tail(8)
    recent.index = recent.index.strftime("%Y-%m")
    show_table(recent, "Latest reference months: raw data → features → z-scores → composite (each published ~1 month later)",
               dec0=["payrolls"], dec2=[c for c in recent.columns if c != "payrolls"])
    if fred.get("USREC") is not None:
        rec_flag = as_month_end(fred["USREC"]).reindex(labor_score.index).fillna(0)
        print(f"Average score in NBER recession months: {labor_score[rec_flag == 1].mean():+.2f} vs "
              f"{labor_score[rec_flag == 0].mean():+.2f} otherwise")
    RESULTS["E"] = {"latest": valid.iloc[-1], "latest_month": valid.index[-1], "percentile": latest_pct}

### Cell 27 (Markdown)

## Module F — Macro regimes from FRED + BLS + EIA, and how strategies did in each
**The idea.** Asset returns depend on the macro state. Equities do best in expansions, long Treasuries hedge contractions (when the Fed cuts and investors look for safety), and trend filters are supposed to earn their keep in drawn-out downturns. We label every month with a transparent, rules-based **real-time** regime and compare returns across regimes.

**Inputs, as they were knowable at each month-end.**

| Input | Source | Lag applied |
|---|---|---|
| Composite labor score (Module E), 3-month average payroll change, Sahm-rule indicator | BLS | 1 month (2 for the shutdown-delayed Sep and Oct 2025 reports; Oct 2025 is superseded by Nov 2025, released the same day) |
| 10Y−2Y spread, monthly average | FRED | 0 |
| CPI year over year | FRED | 1 month |
| 12-month change in fed funds | FRED | 1 month |
| WTI 12-month % change, monthly average | EIA | 0 |

**Rules (fixed in advance from economic reasoning, not tuned on returns).**
- **Contraction** if any *stress* signal fires: payrolls shrinking (3-month average change < 0), Sahm indicator ≥ 0.5, or labor score ≤ −1.5.
- **Slowdown** if not contraction and either labor score < −0.5 or at least two *warning* signals fire: inverted curve, CPI > 4%, fed funds up > 1 pp in 12 months, oil up > 50% in 12 months.
- **Expansion** otherwise.

**Timing.** The regime known at the end of month *t* is matched to returns in month *t+1*, so it could have been traded. NBER recession months are used only to sanity-check the labels.

**What "contraction" means here.** Every stress signal is a labor-market signal, and labor data lag the business cycle: payrolls keep shrinking and unemployment keeps rising after NBER's trough. So "contraction" catches most NBER recession months, but it also covers the jobless recoveries that follow them, when equities have often already bottomed. The hindsight table at the end of the module shows the gap between a real-time label and the true cycle.

In [ ]:
# ### Cell 28 (Code)
REGIME_RULES = {"contraction_labor": -1.5, "slowdown_labor": -0.5, "sahm_trigger": 0.5,
                "cpi_high": 4.0, "fed_tightening_pp": 1.0, "oil_shock_pct": 50.0}
REGIME_ORDER = ["expansion", "slowdown", "contraction"]
REGIME_COLORS = {"expansion": "rgba(44,160,44,0.16)", "slowdown": "rgba(255,165,0,0.22)", "contraction": "rgba(214,39,40,0.24)"}


def build_regime_inputs():
    """Every column at month-end t holds only what had been published by t (features are computed on the
    reference-month index first, then re-stamped at their release month)."""
    lag, overrides = RELEASE_LAG_MONTHS, RELEASE_DATE_OVERRIDES
    labor_pit = lambda s: point_in_time(s, lag["labor"], overrides.get("labor"))
    wti = eia["wti"] if eia.get("wti") is not None else pd.Series(dtype=float, index=pd.DatetimeIndex([]))
    wti_m = wti.where(wti > 0).dropna().resample("ME").mean()
    return pd.DataFrame({
        "labor_score": labor_pit(labor_score),
        "payroll_3m_avg_chg": labor_pit(labor_features["payroll_3m_avg_chg"]),
        "sahm": labor_pit(sahm_indicator(labor_raw["unemployment"])),
        "curve_10y2y": point_in_time(fred["T10Y2Y"].dropna().resample("ME").mean(), lag["curve"]),
        "cpi_yoy": point_in_time(pct_change(as_month_end(fred["CPIAUCSL"]), 12) * 100, lag["cpi"], overrides.get("cpi")),
        "fed_funds_12m_chg": point_in_time(as_month_end(fred["FEDFUNDS"]).diff(12).round(6), lag["fed_funds"]),
        "oil_yoy": point_in_time(pct_change(wti_m, 12) * 100, lag["oil"]),
    })


def classify_regimes(x, rules=REGIME_RULES):
    stress = pd.DataFrame({"payrolls shrinking": x["payroll_3m_avg_chg"] < 0,
                           "Sahm trigger": x["sahm"] >= rules["sahm_trigger"],
                           "labor score very weak": x["labor_score"] <= rules["contraction_labor"]})
    warning = pd.DataFrame({"curve inverted": x["curve_10y2y"] < 0,
                            "CPI high": x["cpi_yoy"] > rules["cpi_high"],
                            "Fed tightening": x["fed_funds_12m_chg"] > rules["fed_tightening_pp"],
                            "oil shock": x["oil_yoy"] > rules["oil_shock_pct"]})
    contraction = stress.any(axis=1)
    slowdown = (x["labor_score"] < rules["slowdown_labor"]) | (warning.sum(axis=1) >= 2)
    labels = np.select([contraction, slowdown], ["contraction", "slowdown"], default="expansion")
    regime = pd.Series(labels, index=x.index, name="regime").where(x[["labor_score", "payroll_3m_avg_chg"]].notna().all(axis=1))
    return regime.dropna(), stress, warning


needed = [labor_raw is not None and "labor_score" in globals(), fred.get("CPIAUCSL") is not None,
          fred.get("T10Y2Y") is not None, fred.get("FEDFUNDS") is not None]
if not all(needed):
    print("⚠️ Missing FRED/BLS inputs — Module F skipped.")
else:
    regime_inputs = build_regime_inputs()
    regime, stress_flags, warning_flags = classify_regimes(regime_inputs)
    regime_runs = {r: true_runs(regime == r) for r in REGIME_ORDER}

    run_lengths = pd.Series({r: np.mean([len(regime.loc[a:b]) for a, b in regime_runs[r]]) if regime_runs[r] else np.nan
                             for r in REGIME_ORDER})
    usrec_m = as_month_end(fred["USREC"]).reindex(regime.index) if fred.get("USREC") is not None else pd.Series(np.nan, index=regime.index)
    regime_summary = pd.DataFrame({"months": regime.value_counts().reindex(REGIME_ORDER),
                                   "share of months": regime.value_counts(normalize=True).reindex(REGIME_ORDER),
                                   "avg spell (months)": run_lengths,
                                   "share in NBER recession": usrec_m.groupby(regime).mean().reindex(REGIME_ORDER)})
    show_table(regime_summary, f"Real-time macro regimes {regime.index[0]:%Y-%m} → {regime.index[-1]:%Y-%m}",
               pct=["share of months", "share in NBER recession"], dec0=["months"], dec2=["avg spell (months)"])

    nber_months = usrec_m == 1
    caught = regime[nber_months].isin(["slowdown", "contraction"]).mean()
    caught_c = (regime[nber_months] == "contraction").mean()
    contraction_precision = usrec_m[regime == "contraction"].mean()  # share of contraction months that were NBER recessions
    print(f"NBER recession months labelled contraction: {caught_c:.0%}; contraction or slowdown: {caught:.0%} "
          f"(n = {int(nber_months.sum())}). Current label ({regime.index[-1]:%Y-%m}): {regime.iloc[-1].upper()}")

    latest_inputs = regime_inputs.dropna(how="all").tail(6)
    latest_inputs.index = latest_inputs.index.strftime("%Y-%m")
    show_table(latest_inputs.assign(regime=regime.reindex(regime_inputs.dropna(how="all").tail(6).index).to_numpy()).T,
               "Regime inputs at recent month-ends (already lagged to what was public then)",
               dec2=list(latest_inputs.T.columns))

In [ ]:
# ### Cell 29 (Code)
def regime_performance(monthly_returns, regime_lagged, rf_monthly):
    rows = []
    for reg in REGIME_ORDER:
        sub = monthly_returns[regime_lagged == reg]
        for asset in monthly_returns:
            r = sub[asset].dropna()
            if len(r) < 3:
                continue
            excess = r - rf_monthly.reindex(r.index).fillna(0.0)
            rows.append({"regime": reg, "asset": asset, "months": len(r),
                         "ann. return": (1 + r).prod() ** (12 / len(r)) - 1, "ann. vol": r.std() * np.sqrt(12),
                         "Sharpe": excess.mean() / excess.std() * np.sqrt(12) if excess.std() > 0 else np.nan,
                         "hit rate": (r > 0).mean(), "worst month": r.min()})
    return pd.DataFrame(rows)


def sharpe_gap_ci(a, b, rf, n_boot=2000, seed=0):
    """90% bootstrap interval for Sharpe(a) − Sharpe(b), resampling the same months for both (paired)."""
    ex = pd.concat([a, b], axis=1).dropna().sub(rf.reindex(a.index).fillna(0.0), axis=0).dropna()
    sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)
    rng = np.random.default_rng(seed)
    draws = [sharpe(s.iloc[:, 0]) - sharpe(s.iloc[:, 1])
             for s in (ex.iloc[rng.integers(0, len(ex), len(ex))] for _ in range(n_boot))]
    return np.nanpercentile(draws, [5, 95]), len(ex)


STRATEGY_NAME = f"SMA {FAST_MA}/{SLOW_MA} strategy"

if "regime" not in globals() or "A" not in RESULTS:
    print("⚠️ Regimes or Module A backtest unavailable — performance-by-regime skipped.")
else:
    bench = [t for t in ("SPY", "QQQ", "TLT") if t in close]  # a benchmark whose download failed is left out
    if len(bench) < 3:
        print(f"⚠️ Benchmarks missing: {sorted({'SPY', 'QQQ', 'TLT'} - set(bench))}")
    month_close = close[bench].resample("ME").last()
    monthly = pd.concat({STRATEGY_NAME: (1 + bt["strategy"]).resample("ME").prod() - 1,
                         **{("Buy & hold SPY" if t == "SPY" else t): month_close[t] / month_close[t].shift(1) - 1
                            for t in bench}}, axis=1, sort=True)
    first_full_month = bt.index[0] + pd.offsets.MonthEnd(0) + pd.offsets.MonthEnd(1)  # skip the partial start month
    last_full_month = END_DATE - pd.offsets.MonthEnd(1)
    monthly = monthly.loc[first_full_month:last_full_month]
    regime_next = regime.shift(1, freq="ME").reindex(monthly.index)  # label known at end of t-1 → return in month t
    monthly, regime_next = monthly[regime_next.notna()], regime_next.dropna()
    rf_m = as_month_end(fred["FEDFUNDS"]) / 100 / 12

    # Each asset uses every month it has (TLT starts 2002-07), so the strategy and SPY keep the 2001 recession.
    perf = regime_performance(monthly, regime_next, rf_m)
    wide = perf.pivot(index="asset", columns="regime", values=["ann. return", "Sharpe", "hit rate", "worst month"])
    wide = wide.reindex(columns=[(m, r) for m in ["ann. return", "Sharpe", "hit rate", "worst month"] for r in REGIME_ORDER])
    wide = wide.reindex(monthly.columns)
    wide.insert(0, ("months", "total"), monthly.count().reindex(wide.index))
    fmt = {col: "{:.2f}" if col[0] == "Sharpe" else "{:.0f}" if col[0] == "months" else "{:.1%}" for col in wide.columns}
    counts = regime_next.value_counts().reindex(REGIME_ORDER)
    starts = {c: monthly[c].first_valid_index() for c in monthly}
    late = [f"{c} from {d:%Y-%m}" for c, d in starts.items() if d > monthly.index[0]]
    display(wide.style.format(fmt, na_rep="—").set_caption(
        f"Performance by regime (label at end of t−1 → return in month t), {monthly.index[0]:%Y-%m} → {monthly.index[-1]:%Y-%m}"
        + (f" ({'; '.join(late)})" if late else "") + " · " + ", ".join(f"{r}: {n} months" for r, n in counts.items())))

    (ci_lo, ci_hi), n_ci = sharpe_gap_ci(monthly.loc[regime_next == "contraction", STRATEGY_NAME],
                                         monthly.loc[regime_next == "contraction", "Buy & hold SPY"], rf_m)
    print(f"Contraction months: Sharpe gap (strategy − buy & hold SPY) 90% bootstrap interval [{ci_lo:+.2f}, {ci_hi:+.2f}], "
          f"n = {n_ci} months; {'excludes' if ci_lo > 0 or ci_hi < 0 else 'includes'} zero.")

    # Hindsight benchmark: the same assets split by NBER recession months (not knowable in real time)
    if fred.get("USREC") is not None:
        nber_flag = as_month_end(fred["USREC"]).reindex(monthly.index).fillna(0)
        rec, exp_ = monthly[nber_flag == 1], monthly[nber_flag == 0]
        hindsight = pd.DataFrame({"months in NBER recession": rec.count(),
                                  "ann. return, NBER recession": (1 + rec).prod() ** (12 / rec.count().clip(lower=1)) - 1,
                                  "ann. return, NBER expansion": (1 + exp_).prod() ** (12 / exp_.count().clip(lower=1)) - 1})
        show_table(hindsight, "Hindsight comparison: NBER recession months vs the rest (not knowable in real time; "
                              "shows how far a real-time label lags the true cycle)",
                   dec0=["months in NBER recession"], pct=["ann. return, NBER recession", "ann. return, NBER expansion"])

    fig = make_subplots(rows=1, cols=2, subplot_titles=["Annualised return in regime", "Sharpe ratio in regime"])
    for asset, color in zip(monthly.columns, ["#1f77b4", "#7f7f7f", "#9467bd", "#2ca02c"]):
        sub = perf[perf["asset"] == asset].set_index("regime").reindex(REGIME_ORDER)
        fig.add_trace(go.Bar(x=REGIME_ORDER, y=sub["ann. return"], name=asset, marker_color=color), row=1, col=1)
        fig.add_trace(go.Bar(x=REGIME_ORDER, y=sub["Sharpe"], name=asset, marker_color=color, showlegend=False), row=1, col=2)
    fig.update_yaxes(tickformat=".0%", row=1, col=1)
    fig.update_layout(barmode="group", height=460, title="Strategy and asset performance by real-time macro regime",
                      legend=dict(orientation="h", yanchor="top", y=-0.15))
    fig.show()

    spy_m = month_close["SPY"].loc[monthly.index[0]:]
    fig = go.Figure()
    for reg in REGIME_ORDER:
        shade(fig, [(a - pd.offsets.MonthBegin(1), b) for a, b in true_runs(regime_next == reg)], color=REGIME_COLORS[reg])
        add_legend_swatch(fig, reg, REGIME_COLORS[reg].replace("0.16", "0.5").replace("0.22", "0.5").replace("0.24", "0.5"))
    fig.add_trace(go.Scatter(x=spy_m.index, y=spy_m, name="SPY (adjusted, log scale)", line=dict(color="black", width=1.6)))
    for a, b in USREC_PERIODS:
        if b >= spy_m.index[0]:
            fig.add_vrect(x0=a, x1=b, line=dict(color="rgba(80,80,80,0.9)", width=1, dash="dot"), fillcolor="rgba(0,0,0,0)")
    fig.update_xaxes(type="date")
    fig.update_yaxes(type="log", title="SPY")
    fig.update_layout(title="Regime timeline (background = regime applied to each month; dotted boxes = NBER recessions)",
                      height=440, legend=dict(orientation="h", y=1.1))
    fig.show()

    get = lambda asset, reg, m="ann. return": perf.set_index(["asset", "regime"])[m].get((asset, reg), np.nan)
    RESULTS["F"] = {"months": len(monthly), "counts": counts.to_dict(), "nber_caught": caught, "nber_contraction": caught_c,
                    "contraction_precision": contraction_precision, "sharpe_gap_ci": (ci_lo, ci_hi), "ci_months": n_ci,
                    "start": monthly.index[0], "end": monthly.index[-1],
                    "current": regime.iloc[-1], "current_month": regime.index[-1],
                    "ret": {(a, r): get(a, r) for a in monthly.columns for r in REGIME_ORDER},
                    "sharpe": {(a, r): get(a, r, "Sharpe") for a in monthly.columns for r in REGIME_ORDER}}

### Cell 30 (Markdown)

## Key findings
*From the reference run on 2026-10-19 (prices through 2026-10-16) with **no API keys**, so every source used its keyless fallback. Your numbers will move as data update. The resume-bullet cell at the end recomputes the headline figures from your own run.*

1. **The trend filter cut risk, not return (Module A).** From October 2000 to October 2026, the 50/200-day crossover (signal at the close, fill at the next open, 5 bps a trade) compounded at **8.9% a year vs 8.7%** for buy-and-hold SPY. Volatility was 13.0% vs 19.1%, the **Sharpe ratio 0.57 vs 0.43**, and the **maximum drawdown −34% vs −55%**. The strategy was invested 73% of the time and traded about once a year (13 round trips, 10 of them winners), so costs barely matter: it still matches buy-and-hold at 25 bps a trade. It avoided most of 2000–02 and 2008. Its worst drawdown came in the V-shaped 2020 crash, which it exited after the bottom.
2. **The yield curve warned early, with long and variable lags (Module B).** Five of the six recessions since 1980 followed a sustained 10Y–2Y inversion. Leads ranged from 11 to 23 months, with a median of **17 months**. The exception was 2020, a pandemic shock preceded only by a few days of inversion in August 2019. The 2022–24 inversion (26 months, deepest −0.93 pp) is the longest on record and has **not** been followed by an NBER recession. The curve has been positive for about two years (+0.38 pp). CPI inflation is 3.7% year over year, unemployment 4.2%, and the Sahm indicator 0.00.
3. **Crypto screen (Module C).** Excluding stablecoins, tokenized funds and gold, and coins under $1 bn market cap or $20 mn daily volume leaves 44 of the top 100. Momentum and turnover rankings change completely from day to day. This is a live snapshot screen, not a tested signal.
4. **Energy stocks track oil, not gas (Module D).** Since 2008, daily correlation with WTI is about **0.5** (0.59 on weekly returns) for XLE and XOM, against about **0.04** with Henry Hub gas. The 1-year rolling oil correlation has ranged from 0.16 to 0.79. Crude-inventory surprises move energy stocks **on release day** (XLE −15 bps per 1σ build, t = −3.2) but **do not predict the next week** (t = 1.2, p = 0.22; raw changes t = 1.1). The sign of the next-week effect even flips between the two halves of the sample. The report is priced in quickly.
5. **The labor score separates recessions (Module E).** The composite averages **−1.72 in NBER recession months vs −0.12 otherwise**. The latest reading (September 2026 data) is −0.55, the 30th percentile of history: softer momentum, not recessionary (payrolls +51k a month over three months, unemployment flat at 4.2%).
6. **Real-time regimes lag the cycle, and that shapes the results (Module F).** Since 1979 the rules label 48% of months expansion, 26% slowdown and 26% contraction. They put **86% of NBER recession months in contraction** (98% in contraction or slowdown). But only 34% of contraction months were NBER recessions, because labor data keep deteriorating through jobless recoveries. Measured on the following month (2000–2026):
   - **SMA strategy:** in contraction it had a **Sharpe of 1.11 vs 0.30** for buy-and-hold SPY. The 90% bootstrap interval for the gap, +0.21 to +1.44, excludes zero.
   - **SPY:** earned only 5.4% a year in contraction, against 7.5% in expansion and 13.9% in slowdown.
   - **TLT:** returned **+9.3%** annualized in actual NBER recession months, but only +0.6% in real-time contraction months, which include recoveries when yields rose.

   The current label is **slowdown**: the labor score is below −0.5, and oil is up 62% year over year.

### Cell 31 (Markdown)

## Limitations
- **Revised data, not vintages.** FRED and BLS serve today's revised numbers. Payrolls in particular are revised for months and re-benchmarked every February, so the labor score and regimes look cleaner here than they did in real time. A strict test would pull ALFRED vintages.
- **Release lags are approximations.** Whole-month lags roughly match CPI and jobs-report timing, but exact release dates vary (the 2025 shutdown delayed releases and left October 2025 household-survey data uncollected; those months are NaN and the score uses the components it has).
- **Backtest realism.** Next-open fills at a flat cost ignore opening-auction slippage, taxes and borrow/margin, and assume fractional, frictionless cash. 50/200 is the textbook setting rather than an optimised one, but it is famous partly *because* it worked, which is a form of hindsight.
- **Price data.** Alpha Vantage's free tier doesn't serve adjusted daily history, so the default run uses yfinance's back-adjusted (dividend-reinvested) prices: an unofficial API whose history can be revised. The equity sample starts in 2000, so the backtest covers three bear markets (2000–02, 2007–09, 2022) and one V-shaped crash (2020). That is a small number of independent episodes. Yahoo data are for personal research use under Yahoo's terms.
- **Energy study.** The surprise proxy is a seasonal norm, not the analyst consensus, and the American Petroleum Institute's Tuesday-evening survey front-runs part of the EIA news. Spot prices and equity closes aren't synchronous. Quintile cut-offs are in-sample.
- **Regimes.** Few contraction months means wide uncertainty in per-regime statistics (no significance tests are claimed). Thresholds were set by economic reasoning, but the researcher has seen these decades, so some implicit snooping is unavoidable.
- **Crypto.** A single snapshot ranking from the top-100 by today's market cap is survivorship-biased and is a screen, not evidence of returns.

In [ ]:
# ### Cell 32 (Code)
def fmt_pct(x, signed=False):
    return "n/a" if x is None or pd.isna(x) else (f"{x:+.1%}" if signed else f"{x:.1%}")


dated = provenance[provenance["first"].notna()]
families = {"Alpha Vantage": "Alpha Vantage", "yfinance": "yfinance", "FRED": "FRED", "EIA": "EIA", "BLS": "BLS",
            "CoinGecko": "CoinGecko"}
sources_used = [name for key, name in families.items() if provenance["source"].str.contains(key).any()]
span = f"{pd.to_datetime(dated['first']).min():%Y}–{pd.to_datetime(dated['last']).max():%Y}"
bullets = [f"Built a Python research pipeline in Google Colab that merges {len(dated)} daily, weekly and monthly series "
           f"plus a live crypto top-100 snapshot from {len(sources_used)} free data sources ({', '.join(sources_used)}; "
           f"{span}), with disk caching, exponential-backoff retries, per-source keyless fallbacks and release-date-aware "
           "alignment to prevent look-ahead bias."]
if "A" in RESULTS:
    a = RESULTS["A"]
    s, b = a["strategy"], a["buy_hold"]
    bullets.append(f"Backtested a 50/200-day SMA crossover on SPY over {a['years']:.0f} years with next-open execution and "
                   f"{a['cost_bps']:g} bps costs: {fmt_pct(s['CAGR'])} CAGR with a {fmt_pct(s['Max drawdown'])} max drawdown vs "
                   f"{fmt_pct(b['CAGR'])} / {fmt_pct(b['Max drawdown'])} for buy-and-hold (Sharpe {s['Sharpe']:.2f} vs {b['Sharpe']:.2f}).")
if "F" in RESULTS:
    f = RESULTS["F"]
    lo, hi = f["sharpe_gap_ci"]
    bullets.append(f"Designed a rules-based, point-in-time macro regime model from FRED, BLS and EIA data (expanding robust "
                   f"z-scores, release-date-aware lags) that labeled {fmt_pct(f['nber_contraction'])} of NBER recession months "
                   f"as contraction ({fmt_pct(f['contraction_precision'])} of contraction labels fell in NBER recessions); in "
                   f"contraction months the SMA strategy's Sharpe was {f['sharpe'][(STRATEGY_NAME, 'contraction')]:.2f} vs "
                   f"{f['sharpe'][('Buy & hold SPY', 'contraction')]:.2f} for SPY (90% bootstrap CI of the gap "
                   f"{lo:+.2f} to {hi:+.2f}, n = {f['ci_months']} months).")
# Fallbacks so three bullets still print when a data outage skipped Module A or F
if "D_inv" in RESULTS:
    d = RESULTS["D_inv"]
    bullets.append(f"Tested {d['n']:,} weekly EIA crude-inventory reports with HAC regressions: surprises moved XLE on release day "
                   f"(t = {d['xle_react_t']:.1f}) but did not predict next-week returns (t = {d['xle_t']:.1f}).")
if "D_corr" in RESULTS:
    c = RESULTS["D_corr"]
    bullets.append(f"Measured energy equities' commodity exposure with rolling correlations: XLE tracks WTI (daily corr "
                   f"{c['wti_xle']:.2f}, weekly {c['wti_xle_weekly']:.2f}) but not Henry Hub gas ({c['hh_xle']:.2f}).")
if "E" in RESULTS:
    e = RESULTS["E"]
    bullets.append(f"Built a composite BLS labor-market score from expanding robust z-scores of payroll growth, unemployment "
                   f"change and wage growth; latest reading {e['latest']:+.2f} ({e['percentile'] * 100:.0f}th percentile of history).")

print("Resume bullets (numbers come from this run):\n")
for line in bullets[:3]:
    print(f"• {line}\n")